In [ ]:
# 一次执行完成训练环境和资产准备；不会自动训练。
from pathlib import Path
import subprocess, sys, os, shutil, traceback, json
from datetime import datetime, timezone
WORKDIR = Path('/content/smolvla-training')  # AutoDL可改为自己的持久化目录
PROFILE = 'gpu'
MOUNT_DRIVE = True  # 非Colab自动跳过；已有挂载不会重复授权
PREPARE_ENV = not ((WORKDIR / 'environment.json').is_file() and (WORKDIR / 'venv/bin/python').is_file() and json.loads((WORKDIR / 'environment.json').read_text()).get('assets_ready', False))  # 新运行时/未完成准备时安装，已完成时只同步
WORKDIR.mkdir(parents=True, exist_ok=True)
SOURCE = '"""训练准备、CPU LoRA机制实验、官方SmolVLA训练；各阶段显式启动。"""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import ExitStack\nfrom datetime import datetime, timezone\nimport hashlib\nimport inspect\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport traceback\n\nLEROBOT_SHA = "7e241bd630a3719a56157a497ce5d08f244784f1"\nBASE_ID = "lerobot/smolvla_base"\nBASE_SHA = "5e8d12a6e2975b0e5e5fce7c8caf47c371d257b6"\nDATA_ID = "HuggingFaceVLA/libero"\nDATA_SHA = "86958911c0f959db2bbbdb107eb3e17c5f9c798e"\nTRAIN_EPISODES = (813, 818)\nHOLDOUT_EPISODES = (825,)\n# 此固定revision的元信息data/file_index与实际分片不符；以下文件已按物理episode列核对。\nDATA_FILES = tuple(f"data/chunk-000/file-{i:03d}.parquet" for i in (213, 214, 215))\nVLM_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"\nPINS = ["peft==0.21.0", "accelerate==1.14.0", "transformers==5.5.4", "tokenizers==0.22.2", "numpy==2.1.3"]\nMANAGED_TORCH_PINS = ["torch==2.11.0", "torchvision==0.26.0", "torchaudio==2.11.0"]\nMANAGED_TORCH_INDEX = "https://download.pytorch.org/whl/cu128"\n\n\nclass BatchInspectionComplete(Exception):\n    """主动结束batch检查，不代表训练失败。"""\n\n\nclass PracticeStepComplete(Exception):\n    """用户的单batch更新已运行，不作为官方训练器的完整checkpoint。"""\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef digest(path):\n    value = hashlib.sha256()\n    with Path(path).open("rb") as file:\n        for chunk in iter(lambda: file.read(1024 * 1024), b""):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef redact(text):\n    text = re.sub(r"(?:hf_|ghp_)[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}", "<TOKEN_REDACTED>", text)\n    return re.sub(r"(https?://)[^/@\\s]+:[^/@\\s]+@", r"\\1<REDACTED>@", text)\n\n\nclass Log:\n    def __init__(self, directory):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.path = self.directory / "run.log"\n\n    def say(self, message):\n        line = redact(str(message))\n        print(line, flush=True)\n        with self.path.open("a", encoding="utf-8") as file:\n            file.write(line + "\\n")\n\n    def run(self, command, *, env=None):\n        self.say("运行：" + json.dumps([str(x) for x in command], ensure_ascii=False))\n        with subprocess.Popen([str(x) for x in command], stdout=subprocess.PIPE,\n                              stderr=subprocess.STDOUT, text=True, errors="replace", env=env) as process:\n            for line in process.stdout:\n                self.say(line.rstrip())\n            code = process.wait()\n        if code:\n            raise RuntimeError(f"子进程退出码{code}；完整日志：{self.path}")\n\n\nclass Tee:\n    def __init__(self, terminal, file):\n        self.terminal, self.file = terminal, file\n\n    def write(self, message):\n        message = redact(message)\n        self.terminal.write(message)\n        self.file.write(message)\n        self.file.flush()\n        return len(message)\n\n    def flush(self):\n        self.terminal.flush()\n        self.file.flush()\n\n    def isatty(self):\n        return False\n\n\ndef resources():\n    memory = {}\n    if Path("/proc/meminfo").exists():\n        for line in Path("/proc/meminfo").read_text().splitlines():\n            key, value = line.split(":", 1)\n            if key in ["MemTotal", "MemAvailable", "SwapFree"]:\n                memory[key + "_GiB"] = round(int(value.split()[0]) / 2**20, 2)\n    return {"python": sys.version, "executable": sys.executable,\n            "cpu_threads": os.cpu_count(), "memory": memory}\n\n\ndef runtime_env(workdir):\n    env = os.environ.copy()\n    env.update({"HF_HOME": str(workdir / "hf"), "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",\n                "HF_HUB_DISABLE_TELEMETRY": "1", "TOKENIZERS_PARALLELISM": "false",\n                "PYTHONUNBUFFERED": "1", "PIP_CACHE_DIR": str(workdir / "pip-cache"),\n                "PIP_DISABLE_PIP_VERSION_CHECK": "1",\n                # Notebook的inline后端模块不一定装在隔离venv；CLI只需无显示器绘图。\n                "MPLBACKEND": "Agg"})\n    # 不继承Notebook中临时拼接的另一份LeRobot源码。\n    env.pop("PYTHONPATH", None)\n    return env\n\n\ndef check_persistent_path(path):\n    if path and str(Path(path).absolute()).startswith("/content/drive/") and not os.path.ismount("/content/drive"):\n        raise RuntimeError("没有检测到/content/drive挂载；拒绝把普通本地目录标作云盘归档。")\n\n\ndef ensure_environment(environment, log, env):\n    """不依赖宿主ensurepip；重试时修复已有Python但没有pip的半成品。"""\n    environment = Path(environment).absolute()\n    python = environment / "bin/python"\n    if not python.exists():\n        log.run([sys.executable, "-m", "venv", "--without-pip", environment], env=env)\n    probe = subprocess.run(\n        [str(python), "-c", "import sys,json;print(json.dumps([sys.prefix,list(sys.version_info[:2])]))"],\n        text=True, capture_output=True, env=env,\n    )\n    if probe.returncode:\n        log.say(probe.stdout + probe.stderr)\n        raise RuntimeError(f"已有环境的Python无法运行，保留现场：{environment}")\n    prefix, py_version = json.loads(probe.stdout)\n    if Path(prefix).resolve() != environment.resolve() or py_version != list(sys.version_info[:2]):\n        raise RuntimeError(f"已有Python属于其他环境或版本，保留现场：{environment}")\n    pip_probe = subprocess.run([str(python), "-m", "pip", "--version"],\n                               text=True, capture_output=True, env=env)\n    if pip_probe.returncode:\n        log.say("环境的Python已存在但pip未就绪，开始补齐；不删除已有文件。")\n        host = subprocess.run([sys.executable, "-c", "import pip;print(pip.__version__)"],\n                              text=True, capture_output=True, env=env)\n        match = re.match(r"(\\d+)\\.(\\d+)", host.stdout.strip())\n        if host.returncode == 0 and match and tuple(map(int, match.groups())) >= (22, 3):\n            # --python指定安装目标，不向Colab全局环境安装/升级依赖。\n            log.run([sys.executable, "-m", "pip", "--python", python,\n                     "install", "pip==25.0.1"], env=env)\n        else:\n            # 本机某些uv环境没有pip，但底层Python的ensurepip可用。\n            log.say("宿主没有可用的pip>=22.3；尝试环境内ensurepip，保留完整诊断输出。")\n            log.run([python, "-m", "ensurepip", "--upgrade", "--default-pip"], env=env)\n    log.run([python, "-m", "pip", "--version"], env=env)\n    return python\n\n\ndef torch_version_supported(version):\n    match = re.match(r"^(\\d+)\\.(\\d+)", version)\n    return bool(match and (2, 7) <= tuple(map(int, match.groups())) < (2, 12))\n\n\ndef prepare_with_managed_torch(args, log):\n    """宿主Torch不兼容时，用独立CUDA wheel环境启动prepare；不改Colab宿主。\n\n    安装和导入发生在新解释器，不会在当前已import的Torch进程内降级。\n    正式venv只链接此环境里的native包；不再借用不兼容的宿主native包。\n    """\n    workdir = args.workdir\n    env = runtime_env(workdir)\n    python = ensure_environment(workdir / "torch-runtime-2.11-cu128", log, env)\n    probe_code = (\n        "import json,torch,torchvision,torchaudio;"\n        "assert torch.__version__.split(\'+\')[0]==\'2.11.0\';"\n        "assert torchvision.__version__.split(\'+\')[0]==\'0.26.0\';"\n        "assert torchaudio.__version__.split(\'+\')[0]==\'2.11.0\';"\n        "assert torch.cuda.is_available(), \'独立环境CUDA不可用\';"\n        "x=torch.ones(2,device=\'cuda\');assert x.sum().item()==2;"\n        "torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]],device=\'cuda\'),torch.tensor([1.],device=\'cuda\'),.5);"\n        "print(json.dumps({\'torch\':torch.__version__,\'torchvision\':torchvision.__version__,"\n        "\'cuda\':torch.version.cuda,\'gpu\':torch.cuda.get_device_name(0)}))"\n    )\n    probe = subprocess.run([str(python), "-c", probe_code], env=env, capture_output=True, text=True)\n    if probe.returncode:\n        log.say("宿主Torch不符合固定LeRobot约束；在独立torch-runtime安装官方2.11.0/cu128组合。宿主不变。")\n        log.say("已有独立环境检查：" + probe.stderr[-2000:])\n        if shutil.disk_usage(workdir).free < 8 * 2**30:\n            raise RuntimeError("独立CUDA wheel安装至少保留8GiB磁盘余量；没有替换宿主Torch")\n        log.run([python, "-m", "pip", "install", "--only-binary=:all:",\n                 "--index-url", MANAGED_TORCH_INDEX, *MANAGED_TORCH_PINS], env=env)\n    log.run([python, "-c", probe_code], env=env)  # 真正做CUDA算子与torchvision ABI检查\n    command = [python, Path(__file__).resolve(), "prepare", "--profile", "gpu", "--workdir", workdir,\n               "--log-root", log.directory / "managed_prepare"]\n    if args.skip_assets:\n        command.append("--skip-assets")\n    # 不传allow-managed-torch：子解释器必须已经兼容，避免失败后无限递归。\n    log.run(command, env=env)\n\n\ndef prepare(args, log):\n    if not (3, 12) <= sys.version_info[:2] < (3, 14):\n        raise RuntimeError("此固定依赖组合使用Python3.12或3.13；不会自动替换系统Python或在其他版本上编译依赖。")\n    workdir = args.workdir\n    previous = json.loads((workdir / "environment.json").read_text()) if (workdir / "environment.json").exists() else {}\n    if previous and previous["profile"] != args.profile:\n        raise RuntimeError("此目录已用于另一准备模式；请另选workdir，不会覆盖已准备环境。")\n    assets_cached = previous.get("assets_ready", False) and all(\n        Path(previous.get(key, "/nonexistent")).is_dir() for key in ["dataset_root", "base_snapshot", "vlm_snapshot"])\n    assets_cached = assets_cached and previous.get("data_files") == list(DATA_FILES) and all(\n        (Path(previous["dataset_root"]) / file).is_file() for file in DATA_FILES)\n    free = shutil.disk_usage(workdir).free / 2**30\n    minimum = 6 if args.profile == "gpu" and not assets_cached else 1\n    log.say({"resources": resources(), "free_disk_GiB": round(free, 2), "profile": args.profile})\n    if free < minimum:\n        raise RuntimeError(f"磁盘余量不足：{free:.2f}GiB；{args.profile}准备至少要求{minimum}GiB。")\n    import torch\n    log.say({"host_torch": torch.__version__, "host_torch_file": torch.__file__,\n             "host_cuda_build": torch.version.cuda, "host_cuda_available": torch.cuda.is_available()})\n    if not torch_version_supported(torch.__version__) or (args.profile == "gpu" and not torch.cuda.is_available()):\n        if args.profile == "gpu" and getattr(args, "allow_managed_torch", False):\n            prepare_with_managed_torch(args, log)\n            return\n        raise RuntimeError(f"实际宿主torch={torch.__version__}，固定LeRobot要求torch>=2.7,<2.12；"\n                           "GPU可传--allow-managed-torch在独立环境安装兼容版本，宿主不会改变。")\n    if args.profile == "gpu" and not torch.cuda.is_available():\n        raise RuntimeError("GPU准备需要已能使用CUDA的PyTorch；在Colab先选择GPU，服务器先安装匹配的CUDA版PyTorch。")\n\n    environment = workdir / "venv"\n    env = runtime_env(workdir)\n    python = ensure_environment(environment, log, env)\n    site = Path(subprocess.check_output([str(python), "-c", "import sysconfig;print(sysconfig.get_path(\'purelib\'))"], text=True).strip())\n    # 只借用现有PyTorch/native wheel，避免复制大文件，也不继承全局diffusers/torchao等可选包。\n    source_site = Path(torch.__file__).resolve().parent.parent\n    names = ["torch", "torchgen", "functorch"]\n    patterns = ["torch-*.dist-info"]\n    if args.profile == "gpu":\n        names += ["torchvision", "torchvision.libs", "torchaudio", "torio", "nvidia", "triton"]\n        patterns += ["torchvision-*.dist-info", "torchaudio-*.dist-info", "nvidia_*.dist-info", "triton-*.dist-info"]\n    sources = [source_site / name for name in names]\n    for pattern in patterns:\n        sources.extend(source_site.glob(pattern))\n    for source in sources:\n        if not source.exists():\n            continue\n        target = site / source.name\n        if target.is_symlink():\n            if target.resolve() != source.resolve():\n                raise RuntimeError(f"已有不同的包链接，停止：{target}")\n        elif not target.exists():\n            target.symlink_to(source, target_is_directory=source.is_dir())\n        else:\n            raise RuntimeError(f"目标已有独立副本，停止以免混用：{target}")\n\n    from importlib.metadata import version, PackageNotFoundError\n    constraints = [f"torch=={version(\'torch\')}"]\n    if args.profile == "gpu":\n        for name in ["torchvision", "torchaudio"]:\n            try:\n                constraints.append(f"{name}=={version(name)}")\n            except PackageNotFoundError:\n                pass\n    constraint_file = workdir / "torch-constraints.txt"\n    constraint_file.write_text("\\n".join(constraints) + "\\n")\n    log.run([python, "-m", "pip", "install", "--constraint", constraint_file,\n             "--report", log.directory / "pip-core-report.json", *PINS], env=env)\n    source = workdir / "lerobot"\n    if args.profile == "gpu":\n        if not (source / ".git").exists():\n            source.mkdir(exist_ok=True)\n            log.run(["git", "init", str(source)])\n            log.run(["git", "-C", source, "remote", "add", "origin", "https://github.com/huggingface/lerobot.git"])\n            log.run(["git", "-C", source, "fetch", "--depth=1", "origin", LEROBOT_SHA])\n            log.run(["git", "-C", source, "checkout", "--detach", "FETCH_HEAD"])\n        actual = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()\n        if actual != LEROBOT_SHA:\n            raise RuntimeError("已有LeRobot源码提交不同；请使用另一工作目录，不会重置现有源码。")\n        import tomllib\n        project = tomllib.loads((source / "pyproject.toml").read_text())\n        dataset_requirements = project["project"]["optional-dependencies"]["dataset"]\n        # 本次是Parquet图像列，视频回退显式使用PyAV；不引入与宿主Torch绑定的可选TorchCodec。\n        dataset_requirements = [r for r in dataset_requirements if not r.startswith("torchcodec") and r != "lerobot[av-dep]"]\n        log.run([python, "-m", "pip", "install", "--constraint", constraint_file,\n                 "--report", log.directory / "pip-lerobot-report.json",\n                 "-e", f"{source}[training,smolvla,peft,av-dep,scipy-dep]", *dataset_requirements, *PINS], env=env)\n    check = "import torch,peft,transformers,accelerate;print(\'torch\',torch.__version__,\'peft\',peft.__version__,\'transformers\',transformers.__version__,\'accelerate\',accelerate.__version__)"\n    if args.profile == "gpu":\n        check += ";from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy;import lerobot.scripts.lerobot_train;print(\'SmolVLA和官方训练入口导入成功\')"\n    log.run([python, "-c", check], env=env)\n    log.run([python, "-m", "pip", "check"], env=env)\n    freeze = subprocess.check_output([str(python), "-m", "pip", "freeze"], text=True, env=env)\n    (log.directory / "pip-freeze.txt").write_text(redact(freeze))\n    manifest = {**previous, "profile": args.profile, "python": str(python), "torch_source": str(source_site),\n                "lerobot_revision": LEROBOT_SHA if args.profile == "gpu" else None, "assets_ready": bool(assets_cached)}\n    if args.profile == "gpu":\n        props = torch.cuda.get_device_properties(0)\n        manifest["gpu"] = {"name": props.name, "memory_GiB": props.total_memory / 2**30,\n                           "capability": list(torch.cuda.get_device_capability(0)), "torch_cuda": torch.version.cuda}\n        log.say(manifest["gpu"])\n    write_json(workdir / "environment.json", manifest)\n    if args.profile == "gpu" and not args.skip_assets and not assets_cached:\n        # 另起已准备的解释器下载，避免当前Notebook的旧包缓存影响结果。\n        log.run([python, Path(__file__).resolve(), "assets", "--workdir", workdir], env=env)\n    elif assets_cached:\n        log.say("复用已记录的模型/数据快照，不重新解析VLM的main。")\n        if args.profile == "gpu":\n            log.run([python, Path(__file__).resolve(), "check-data", "--workdir", workdir], env=env)\n    current_manifest = json.loads((workdir / "environment.json").read_text())\n    if args.profile == "gpu" and current_manifest.get("assets_ready"):\n        # 导入成功不等于配置可从离线缓存加载；实际解析固定SHA的config.json。\n        config_check = (\n            "from lerobot.configs.policies import PreTrainedConfig;"\n            "from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig;"\n            f"c=PreTrainedConfig.from_pretrained({BASE_ID!r},revision={BASE_SHA!r},local_files_only=True);"\n            "print(\'固定SHA的策略配置离线读取成功：\',c.type,\'chunk_size=\',c.chunk_size)"\n        )\n        log.run([python, "-c", config_check], env=env)\n    log.say(f"准备完成。解释器：{python}；不会自动开始训练。")\n\n\ndef assets(args, log):\n    from huggingface_hub import snapshot_download\n    workdir = args.workdir\n    manifest = json.loads((workdir / "environment.json").read_text())\n    dataset = snapshot_download(DATA_ID, repo_type="dataset", revision=DATA_SHA, token=False,\n                                local_dir=workdir / "dataset", allow_patterns=["meta/**", *DATA_FILES])\n    model_files = ["*.json", "*.safetensors", "*.model", "*.txt", "*.tiktoken", "*.jinja"]\n    base = snapshot_download(BASE_ID, revision=BASE_SHA, token=False, allow_patterns=model_files)\n    vlm = snapshot_download(VLM_ID, revision=manifest.get("vlm_resolved_revision") or "main", token=False, allow_patterns=model_files)\n    manifest.update({"assets_ready": False, "dataset_root": dataset, "dataset_revision": DATA_SHA,\n                     "data_files": list(DATA_FILES),\n                     "base_revision": BASE_SHA, "base_snapshot": base,\n                     "vlm_snapshot": vlm, "vlm_resolved_revision": Path(vlm).name})\n    write_json(workdir / "environment.json", manifest)\n    check_data(args, log)\n    manifest["assets_ready"] = True\n    write_json(workdir / "environment.json", manifest)\n    log.say("模型和选定数据文件准备完成，真实样本读取通过；尚未加载完整模型或执行更新。")\n\n\ndef validate_dataset_files(root, episodes=(*TRAIN_EPISODES, *HOLDOUT_EPISODES)):\n    """核对实际数据行与episode元信息；不信任发布的data/file_index。"""\n    import pyarrow.parquet as pq\n    root = Path(root)\n    columns = ["episode_index", "task_index", "index", "frame_index"]\n    selected = {episode: [] for episode in episodes}\n    locations = {episode: [] for episode in episodes}\n    for file in sorted((root / "data").glob("*/*.parquet")):\n        found = set()\n        for row in pq.read_table(file, columns=columns).to_pylist():\n            episode = row["episode_index"]\n            if episode in selected:\n                selected[episode].append(row)\n                found.add(episode)\n        for episode in found:\n            locations[episode].append(str(file.relative_to(root)))\n    meta_files = sorted((root / "meta/episodes").glob("*/*.parquet"))\n    if not meta_files:\n        raise RuntimeError("缺少episode元信息，无法验证帧数和边界。")\n    records = pq.read_table(meta_files, columns=["episode_index", "length", "dataset_from_index", "dataset_to_index"]).to_pylist()\n    meta = {row["episode_index"]: row for row in records}\n    report = {}\n    for episode, rows in selected.items():\n        expected = meta[episode]\n        if len(rows) != expected["length"]:\n            raise RuntimeError(f"episode {episode}实际找到{len(rows)}帧，元信息要求{expected[\'length\']}帧；请修复下载文件，不能进入训练。")\n        if (sorted(row["frame_index"] for row in rows) != list(range(expected["length"]))\n                or sorted(row["index"] for row in rows) != list(range(expected["dataset_from_index"], expected["dataset_to_index"]))\n                or {row["task_index"] for row in rows} != {24}):\n            raise RuntimeError(f"episode {episode}的帧索引、全局索引或任务不符合固定实验配置。")\n        report[str(episode)] = {"frames": len(rows), "files": locations[episode], "task_index": 24}\n    return report\n\n\ndef check_data(args, log):\n    """CPU读取真实示范及50步动作标签，不加载模型、不更新参数。"""\n    import torch\n    from lerobot.datasets.lerobot_dataset import LeRobotDataset\n    manifest = json.loads((args.workdir / "environment.json").read_text())\n    root = Path(manifest["dataset_root"])\n    report = {"episodes": validate_dataset_files(root), "scope": "raw dataset CPU preflight; no policy update"}\n    fps = json.loads((root / "meta/info.json").read_text())["fps"]\n    dataset = LeRobotDataset(DATA_ID, root=root, revision=DATA_SHA, episodes=list(TRAIN_EPISODES),\n                            delta_timestamps={"action": [i / fps for i in range(50)]}, video_backend="pyav")\n    expected_rows = sum(report["episodes"][str(ep)]["frames"] for ep in TRAIN_EPISODES)\n    if len(dataset) != expected_rows:\n        raise RuntimeError("官方读取器的训练帧数不符，停止。")\n    sample = dataset[0]\n    report["train_rows"] = len(dataset)\n    report["first_sample"] = {k: {"shape": list(v.shape), "dtype": str(v.dtype),\n                                  "finite": bool(torch.isfinite(v).all())} for k, v in sample.items() if isinstance(v, torch.Tensor)}\n    report["task"] = sample.get("task")\n    if not all(value["finite"] for value in report["first_sample"].values()):\n        raise RuntimeError("真实示范样本包含非有限数，停止。")\n    if sample["action"].shape != (50, 7) or sample["observation.state"].shape != (8,):\n        raise RuntimeError("原始示范的状态或50步动作维度不符合本实验。")\n    boundary = dataset[report["episodes"][str(TRAIN_EPISODES[0])]["frames"] - 1]\n    report["last_frame_action_padding"] = int(boundary["action_is_pad"].sum())\n    if report["last_frame_action_padding"] != 49:\n        raise RuntimeError("episode末帧的动作padding不符合50步标签边界。")\n    write_json(args.workdir / "dataset_check.json", report)\n    log.say(report)\n\n\ndef cpu_lab(args, log):\n    import torch\n    from torch import nn\n    from peft import LoraConfig, PeftModel, get_peft_model\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    torch.manual_seed(7)\n    output = args.workdir / "cpu_lora"\n    if output.exists():\n        raise FileExistsError(f"保留已有CPU实验：{output}；另选workdir重做。")\n    output.mkdir()\n\n    class ProjectionPair(nn.Module):\n        def __init__(self):\n            super().__init__()\n            self.q_proj = nn.Linear(720, 960, bias=False)\n            self.o_proj = nn.Linear(960, 720, bias=False)\n\n        def forward(self, x):\n            return self.o_proj(self.q_proj(x))\n\n    base = ProjectionPair()\n    base_state = {name: p.detach().clone() for name, p in base.state_dict().items()}\n    x = torch.randn(1, 50, 720)\n    with torch.no_grad():\n        original_output = base(x)\n        target = original_output + .1 * x\n    policy = get_peft_model(base, LoraConfig(r=8, lora_alpha=8, target_modules=["q_proj"], lora_dropout=0.0))\n    print("\\n" + "="*50)\n    print("LoRA可训练参数")\n    total_trainable = 0\n\n    for name, param in policy.named_parameters():\n        if param.requires_grad:\n            num_params = param.numel()\n            print(f"参数名称: {name:60s} | 形状: {str(list(param.shape)):20s} | 数量: {num_params}")\n            total_trainable += num_params\n\n    print("-" * 50)\n    print(f"可训练参数总数: {total_trainable}")\n    print(f"目标值: 13440")\n    print(f"是否匹配: {total_trainable == 13440}")\n    print("="*50 + "\\n")\n\n    optimizer = torch.optim.AdamW([p for p in policy.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.0)\n    evidence = {"scope": "CPU synthetic-activation LoRA mechanism experiment, not SmolVLA policy training",\n                "input_shape": list(x.shape), "trainable_parameters": sum(p.numel() for p in policy.parameters() if p.requires_grad),\n                "module_after_injection": str(policy.base_model.model.q_proj), "weight_decay": 0.0, "steps": []}\n    torch.testing.assert_close(policy(x), original_output)\n    policy.train()\n    for step in range(5):\n        optimizer.zero_grad(set_to_none=True)\n        before = {n: p.detach().clone() for n, p in policy.named_parameters() if p.requires_grad}\n        loss = torch.nn.functional.mse_loss(policy(x), target)\n        loss.backward()\n        gradients = {n: float(p.grad.norm()) if p.grad is not None else None for n, p in policy.named_parameters() if p.requires_grad}\n        optimizer.step()\n        deltas = {n: float((p.detach() - before[n]).abs().max()) for n, p in policy.named_parameters() if p.requires_grad}\n        row = {"step": step + 1, "loss": float(loss.detach()), "gradient_norms": gradients, "parameter_max_changes": deltas}\n        evidence["steps"].append(row)\n        log.say(row)\n    for name, weight in base_state.items():\n        current = policy.base_model.model.q_proj.base_layer.weight if name == "q_proj.weight" else policy.base_model.model.o_proj.weight\n        torch.testing.assert_close(current, weight, rtol=0, atol=0)\n    policy.eval()\n    expected = policy(x).detach()\n    with torch.no_grad():\n        projection = policy.base_model.model.q_proj\n        base_q = projection.base_layer(x)\n        low_rank = projection.lora_A["default"](x)\n        correction = projection.lora_B["default"](low_rank) * projection.scaling["default"]\n        adapted_q = projection(x)\n        torch.testing.assert_close(adapted_q, base_q + correction)\n        evidence["projection_example"] = {\n            "scope": "measured CPU projection values with synthetic hidden inputs; not robot actions",\n            "x_shape": list(x.shape), "A_output_shape": list(low_rank.shape), "q_output_shape": list(adapted_q.shape),\n            "base_first3": base_q[0, 0, :3].tolist(), "correction_first3": correction[0, 0, :3].tolist(),\n            "adapted_first3": adapted_q[0, 0, :3].tolist(), "scale": projection.scaling["default"]}\n    torch.save(base_state, output / "base.pt")\n    torch.save(optimizer.state_dict(), output / "optimizer.pt")\n    policy.save_pretrained(output / "adapter")\n    restored_base = ProjectionPair()\n    restored_base.load_state_dict(torch.load(output / "base.pt", weights_only=True))\n    restored = PeftModel.from_pretrained(restored_base, output / "adapter", is_trainable=True)\n    restored.eval()\n    error = float((restored(x).detach() - expected).abs().max())\n    torch.testing.assert_close(restored(x), expected, rtol=1e-5, atol=1e-6)\n    resumed_optimizer = torch.optim.AdamW([p for p in restored.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.0)\n    resumed_optimizer.load_state_dict(torch.load(output / "optimizer.pt", weights_only=True))\n    resumed_optimizer.zero_grad(set_to_none=True)\n    resume_loss = torch.nn.functional.mse_loss(restored(x), target)\n    resume_loss.backward()\n    resumed_optimizer.step()\n    evidence.update({"frozen_base_unchanged": True, "reload_max_error": error,\n                     "optimizer_reloaded_and_one_more_step_completed": True})\n    write_json(output / "evidence.json", evidence)\n    log.say("CPU机制实验完成；冻结、梯度、参数变化、adapter重载与optimizer续一步均已检查。")\n\n\ndef archive_checkpoint(source, destination):\n    source, destination = Path(source), Path(destination)\n    files = {str(p.relative_to(source)): digest(p) for p in source.rglob("*") if p.is_file() and not p.is_symlink()}\n    if not files or not (source / "pretrained_model").is_dir() or not (source / "training_state").is_dir():\n        raise RuntimeError("checkpoint缺少模型或训练状态目录，拒绝标记完整。")\n    if destination.exists():\n        recorded = json.loads((destination / "COMPLETE.json").read_text())\n        if recorded != files or any(not (destination / n).is_file() or digest(destination / n) != h for n, h in files.items()):\n            raise RuntimeError("已有归档不同或不完整，拒绝覆盖。")\n        return\n    staging = destination.with_name(destination.name + ".partial")\n    staging.mkdir(parents=True, exist_ok=True)\n    extras = {str(p.relative_to(staging)) for p in staging.rglob("*") if p.is_file()} - set(files) - {"COMPLETE.json"}\n    if extras:\n        raise RuntimeError("部分归档含其他文件，保留检查，拒绝混合不同checkpoint。")\n    for name, expected in files.items():\n        target = staging / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        if target.exists() and digest(target) != expected:\n            raise RuntimeError(f"部分归档内容不同，保留以检查：{target}")\n        if not target.exists():\n            shutil.copy2(source / name, target)\n        if digest(target) != expected:\n            raise RuntimeError(f"归档校验失败：{name}")\n    write_json(staging / "COMPLETE.json", files)\n    staging.rename(destination)\n\n\ndef save_inspection_images(batch, directory):\n    """导出处理后batch的最后一帧，兼容[B,C,H,W]与[B,T,C,H,W]。"""\n    import torch\n    from PIL import Image\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    files = []\n    for name, value in batch.items():\n        if not name.startswith("observation.images.") or value.ndim not in (4, 5):\n            continue\n        image = value[0, -1] if value.ndim == 5 else value[0]\n        if image.shape[0] != 3:\n            continue\n        rgb = (image.detach().cpu().clamp(0, 1).permute(1, 2, 0) * 255).to(torch.uint8).numpy()\n        path = directory / (name.replace(".", "_") + ".png")\n        Image.fromarray(rgb).save(path)\n        files.append(path.name)\n    return files\n\n\ndef load_policy_config(loader, pretrained_name_or_path, *args, **kwargs):\n    """LeRobot的CLI配置读取未转发pretrained_revision；与预下载的SHA保持一致。"""\n    if str(pretrained_name_or_path) == BASE_ID:\n        if kwargs.get("revision") not in (None, BASE_SHA):\n            raise ValueError("策略配置请求的revision与本实验固定的base SHA不一致。")\n        kwargs["revision"] = BASE_SHA\n        kwargs["local_files_only"] = True\n    return loader(pretrained_name_or_path, *args, **kwargs)\n\n\ndef train(args, log):\n    import gc\n    import torch\n    from unittest.mock import patch\n    practice_step = None\n    if args.practice_step_file:\n        import importlib.util\n        spec = importlib.util.spec_from_file_location("user_training_step", args.practice_step_file)\n        practice_module = importlib.util.module_from_spec(spec)\n        spec.loader.exec_module(practice_module)\n        practice_step = practice_module.train_one_batch\n        if not callable(practice_step):\n            raise TypeError("练习文件必须定义可调用的train_one_batch。")\n    manifest = json.loads((args.workdir / "environment.json").read_text())\n    if not manifest.get("assets_ready") or not torch.cuda.is_available():\n        raise RuntimeError("先完成GPU prepare及资产下载，再启动完整策略训练。")\n    from lerobot.scripts import lerobot_train\n    from lerobot.configs.policies import PreTrainedConfig\n    from lerobot.policies.factory import make_pre_post_processors\n    from transformers import AutoModelForImageTextToText\n    original_config_loader = PreTrainedConfig.from_pretrained\n\n    def load_config(pretrained_name_or_path, *a, **kw):\n        return load_policy_config(original_config_loader, pretrained_name_or_path, *a, **kw)\n\n    log.say({"base_config_revision": BASE_SHA, "base_snapshot": manifest.get("base_snapshot"),\n             "note": "配置、权重使用同一固定SHA；不依赖缓存中的main引用。"})\n    if not Path(lerobot_train.__file__).resolve().is_relative_to((args.workdir / "lerobot").resolve()):\n        raise RuntimeError("导入了另一份LeRobot；请用准备生成的解释器并移除外部PYTHONPATH。")\n    if practice_step:\n        output = (args.archive_dir or args.workdir) / "practice_step"\n    else:\n        output = args.workdir / ("batch_inspection" if args.inspect_only else "train_run")\n    if output.exists():\n        raise FileExistsError("保留已有训练目录；续训须单独核验checkpoint，本入口不会覆盖或假装自动续训。")\n    evidence = {"scope": "user-written one-batch exercise" if practice_step else "official SmolVLA PEFT training",\n                "source_manifest": manifest, "steps": []}\n    original_update = lerobot_train.update_policy\n    original_save = lerobot_train.save_checkpoint\n    original_make = lerobot_train.make_policy\n    holder = {}\n\n    def capture_make(*a, **kw):\n        holder["dataset_meta"] = kw["ds_meta"]\n        return original_make(*a, **kw)\n\n    def audited_update(*positional, **keywords):\n        values = inspect.signature(original_update).bind(*positional, **keywords).arguments\n        policy, optimizer = values["policy"], values["optimizer"]\n        accelerator = values["accelerator"]\n        if accelerator.num_processes != 1:\n            raise RuntimeError("此实践入口只验证单进程训练。")\n        if practice_step and accelerator.mixed_precision != "no":\n            raise RuntimeError("手写单步练习暂不接入AMP/GradScaler；请使用本入口的use_amp=false配置。")\n        unwrapped = accelerator.unwrap_model(policy)\n        holder["policy"] = unwrapped\n        if not evidence.get("trainable"):\n            evidence["trainable"] = {n: list(p.shape) for n, p in policy.named_parameters() if p.requires_grad}\n            evidence["trainable_count"] = sum(p.numel() for p in policy.parameters() if p.requires_grad)\n            evidence["total_count"] = sum(p.numel() for p in policy.parameters())\n            evidence["accelerator_mixed_precision"] = accelerator.mixed_precision\n            dtype_counts = {}\n            for p in policy.parameters():\n                key = str(p.dtype)\n                dtype_counts[key] = dtype_counts.get(key, 0) + p.numel()\n            evidence["parameter_dtype_counts"] = dtype_counts\n            for config in unwrapped.peft_config.values():\n                config.revision = BASE_SHA\n            batch = {n: v.detach().cpu() for n, v in values["batch"].items() if isinstance(v, torch.Tensor)}\n            evidence["first_processed_batch"] = {n: {"shape": list(v.shape), "dtype": str(v.dtype),\n                                                       "finite": bool(torch.isfinite(v).all()),\n                                                       "min": float(v.min()) if v.numel() else None,\n                                                       "max": float(v.max()) if v.numel() else None} for n, v in batch.items()}\n            if not all(v["finite"] for v in evidence["first_processed_batch"].values()):\n                raise RuntimeError("首个处理后batch含非有限数。")\n            output.mkdir(parents=True, exist_ok=True)\n            torch.save(batch, output / "first_processed_batch.pt")\n            if args.inspect_only:\n                evidence["inspection_images"] = save_inspection_images(batch, output)\n                write_json(output / "batch_inspection.json", evidence)\n                raise BatchInspectionComplete()\n        original_step = optimizer.step\n        original_forward = policy.forward\n        row = {"step": len(evidence["steps"]) + 1, "optimizer_step_calls": 0}\n\n        def audited_forward(*a, **kw):\n            result = original_forward(*a, **kw)\n            loss = result[0]\n            row["loss"] = float(loss.detach())\n            if not math.isfinite(row["loss"]) or not loss.requires_grad:\n                raise RuntimeError("loss非有限或没有梯度路径，停止。")\n            return result\n\n        def audited_step(*a, **kw):\n            row["optimizer_step_calls"] += 1\n            if practice_step and row["optimizer_step_calls"] != 1:\n                raise RuntimeError("单batch练习只能调用一次optimizer.step。")\n            before = {n: p.detach().cpu().clone() for n, p in policy.named_parameters() if p.requires_grad}\n            gradients = [p.grad.detach() for p in policy.parameters() if p.requires_grad and p.grad is not None]\n            row["any_nonzero_gradient"] = any(bool(g.abs().max() > 0) for g in gradients)\n            row["all_recorded_gradients_finite"] = all(bool(torch.isfinite(g).all()) for g in gradients)\n            if not gradients or not row["all_recorded_gradients_finite"]:\n                raise RuntimeError("没有adapter梯度或梯度非有限，停止。")\n            result = original_step(*a, **kw)\n            row["any_trainable_parameter_changed"] = any(not torch.equal(before[n], p.detach().cpu()) for n, p in policy.named_parameters() if p.requires_grad)\n            return result\n\n        with patch.object(optimizer, "step", audited_step), patch.object(policy, "forward", audited_forward):\n            if practice_step:\n                row["returned_loss"] = float(practice_step(policy, values["batch"], optimizer))\n                if (row["optimizer_step_calls"] != 1 or "loss" not in row\n                        or not math.isfinite(row["returned_loss"])\n                        or not math.isclose(row["returned_loss"], row["loss"], rel_tol=1e-5, abs_tol=1e-7)\n                        or not row.get("any_trainable_parameter_changed")):\n                    raise RuntimeError("练习未完成有效前向/更新，不能标记通过。")\n                result = None\n            else:\n                result = original_update(*positional, **keywords)\n        row["cuda_peak_allocated_GiB"] = torch.cuda.max_memory_allocated() / 2**30\n        row["cuda_peak_reserved_GiB"] = torch.cuda.max_memory_reserved() / 2**30\n        evidence["steps"].append(row)\n        write_json(output / "training_evidence.json", evidence)\n        log.say(row)\n        if practice_step:\n            unwrapped.save_pretrained(output / "adapter")\n            torch.save(optimizer.state_dict(), output / "optimizer.pt")\n            shutil.copy2(args.practice_step_file, output / "train_one_batch.py")\n            raise PracticeStepComplete()\n        return result\n\n    def saved_checkpoint(*a, **kw):\n        result = original_save(*a, **kw)\n        values = inspect.signature(original_save).bind(*a, **kw).arguments\n        source = Path(values["checkpoint_dir"])\n        holder["checkpoint"] = source\n        holder["postprocessor"] = values["postprocessor"]\n        shutil.copy2(output / "training_evidence.json", source / "training_evidence.json")\n        if args.archive_dir:\n            archive_checkpoint(source, args.archive_dir / "checkpoints" / source.name)\n            log.say(f"完整checkpoint已核对归档：{args.archive_dir / \'checkpoints\' / source.name}")\n        return result\n\n    original_loader = AutoModelForImageTextToText.from_pretrained\n    def load_vlm(*a, **kw):\n        if "torch_dtype" in kw:\n            kw.setdefault("dtype", kw.pop("torch_dtype"))\n        kw.setdefault("token", False)\n        return original_loader(*a, **kw)\n\n    arguments = [f"--policy.path={BASE_ID}", f"--policy.pretrained_revision={BASE_SHA}",\n                 "--policy.input_features=null", "--policy.output_features=null", "--policy.device=cuda",\n                 "--policy.use_amp=false", "--policy.push_to_hub=false", "--policy.optimizer_lr=1e-4",\n                 "--policy.scheduler_warmup_steps=0", f"--dataset.repo_id={DATA_ID}",\n                 f"--dataset.revision={DATA_SHA}", f"--dataset.root={manifest[\'dataset_root\']}",\n                 f"--dataset.episodes={json.dumps(TRAIN_EPISODES)}", "--dataset.video_backend=pyav", "--peft.method_type=LORA",\n                 "--peft.r=8", "--peft.lora_alpha=8", "--batch_size=1", "--num_workers=0",\n                 "--steps=5", "--log_freq=1", "--save_freq=5", "--env_eval_freq=0",\n                 "--wandb.enable=false", f"--output_dir={output}"]\n    with ExitStack() as stack:\n        stack.enter_context(patch.object(PreTrainedConfig, "from_pretrained", staticmethod(load_config)))\n        stack.enter_context(patch.object(lerobot_train, "update_policy", audited_update))\n        stack.enter_context(patch.object(lerobot_train, "save_checkpoint", saved_checkpoint))\n        stack.enter_context(patch.object(lerobot_train, "make_policy", capture_make))\n        stack.enter_context(patch.object(AutoModelForImageTextToText, "from_pretrained", staticmethod(load_vlm)))\n        stack.enter_context(patch.object(sys, "argv", ["lerobot-train", *arguments]))\n        try:\n            lerobot_train.main()\n        except BatchInspectionComplete:\n            log.say(f"真实batch与可训练参数已保存至{output}；尚未backward或optimizer.step。")\n            return\n        except PracticeStepComplete:\n            log.say(f"你的train_one_batch已完成一次真实更新，证据与adapter：{output}；此目录不是官方可续训checkpoint。")\n            return\n    if not any(row.get("any_trainable_parameter_changed") for row in evidence["steps"]):\n        raise RuntimeError("训练返回但未测到参数更新，不能验收为完成训练。")\n    # 保存后的策略重载：同一处理后输入、同一初始噪声，不再重复归一化。\n    batch_cpu = torch.load(output / "first_processed_batch.pt", weights_only=True)\n    policy = holder.pop("policy")\n    policy.eval()\n    noise = torch.randn(len(batch_cpu["observation.state"]), policy.config.chunk_size,\n                        policy.config.max_action_dim, generator=torch.Generator().manual_seed(71))\n    batch = {n: v.to("cuda") for n, v in batch_cpu.items()}\n    with torch.inference_mode():\n        expected = policy.predict_action_chunk(batch, noise=noise.to("cuda")).cpu()\n        repeated = policy.predict_action_chunk(batch, noise=noise.to("cuda")).cpu()\n        expected_action = holder["postprocessor"](expected[:, 0].clone().to("cuda")).cpu()\n    torch.testing.assert_close(repeated, expected, rtol=1e-3, atol=1e-4)\n    del policy, batch, repeated\n    gc.collect()\n    torch.cuda.empty_cache()\n    pretrained = holder["checkpoint"] / "pretrained_model"\n    cfg = PreTrainedConfig.from_pretrained(pretrained)\n    cfg.pretrained_path = pretrained\n    cfg.device = "cuda"\n    with patch.object(AutoModelForImageTextToText, "from_pretrained", staticmethod(load_vlm)):\n        restored = original_make(cfg=cfg, ds_meta=holder["dataset_meta"])\n    restored.eval()\n    _, restored_post = make_pre_post_processors(policy_cfg=restored.config, pretrained_path=pretrained)\n    with torch.inference_mode():\n        actual = restored.predict_action_chunk({n: v.to("cuda") for n, v in batch_cpu.items()}, noise=noise.to("cuda")).cpu()\n        actual_action = restored_post(actual[:, 0].clone().to("cuda")).cpu()\n    torch.testing.assert_close(actual, expected, rtol=1e-3, atol=1e-4)\n    torch.testing.assert_close(actual_action, expected_action, rtol=1e-3, atol=1e-4)\n    reload_evidence = {"fixed_noise_seed": 71, "normalized_chunk_max_error": float((actual - expected).abs().max()),\n                       "postprocessed_first_action_max_error": float((actual_action - expected_action).abs().max()),\n                       "processors_loaded": True, "note": "GPU runtime validation; not task success or exact training-resume validation"}\n    write_json(output / "reload_evidence.json", reload_evidence)\n    if args.archive_dir:\n        write_json(args.archive_dir / "reload_evidence.json", reload_evidence)\n    log.say(reload_evidence)\n    log.say("官方5步训练、固定输入/噪声重载比较已返回；真实任务表现仍需另行评测。")\n\n\ndef export_notebook(args):\n    source = Path(__file__).read_text()\n    practice_source = (Path(__file__).resolve().parents[1] / "studies/policy_generalization/train_one_batch.py").read_text()\n    metric_source = (Path(__file__).resolve().parents[1] / "studies/policy_generalization/evaluation_metrics.py").read_text()\n    eval_source = Path(__file__).with_name("smolvla_eval_tools.py").read_text()\n    sim_source = Path(__file__).with_name("smolvla_libero_tools.py").read_text()\n    cell = ("# 一次执行完成训练环境和资产准备；不会自动训练。\\n"\n            "from pathlib import Path\\nimport subprocess, sys, os, shutil, traceback, json\\nfrom datetime import datetime, timezone\\n"\n            "WORKDIR = Path(\'/content/smolvla-training\')  # AutoDL可改为自己的持久化目录\\nPROFILE = \'gpu\'\\nMOUNT_DRIVE = True  # 非Colab自动跳过；已有挂载不会重复授权\\n"\n            "PREPARE_ENV = not ((WORKDIR / \'environment.json\').is_file() and (WORKDIR / \'venv/bin/python\').is_file() and json.loads((WORKDIR / \'environment.json\').read_text()).get(\'assets_ready\', False))  # 新运行时/未完成准备时安装，已完成时只同步\\n"\n            "WORKDIR.mkdir(parents=True, exist_ok=True)\\n"\n            f"SOURCE = {source!r}\\n"\n            "SCRIPT = WORKDIR / \'smolvla_training_workbench.py\'\\nSCRIPT.write_text(SOURCE)\\n"\n            f"PRACTICE_SOURCE = {practice_source!r}\\n"\n            "PRACTICE_FILE = WORKDIR / \'train_one_batch.py\'\\n"\n            "if not PRACTICE_FILE.exists(): PRACTICE_FILE.write_text(PRACTICE_SOURCE)\\n"\n            "else: print(\'保留已有练习文件：\', PRACTICE_FILE, \'；本地修改后需显式同步此文件。\')\\n"\n            f"METRICS_SOURCE = {metric_source!r}\\nEVAL_SOURCE = {eval_source!r}\\n"\n            "METRICS_FILE = WORKDIR / \'evaluation_metrics.py\'\\n"\n            "if not METRICS_FILE.exists(): METRICS_FILE.write_text(METRICS_SOURCE)\\n"\n            "(WORKDIR / \'smolvla_eval_tools.py\').write_text(EVAL_SOURCE)\\n"\n            f"SIM_SOURCE = {sim_source!r}\\n"\n            "(WORKDIR / \'smolvla_libero_tools.py\').write_text(SIM_SOURCE)\\n"\n            "ARCHIVE = None\\n"\n            "LOG_ROOT = WORKDIR / \'logs\'\\n"\n            "try:\\n"\n            "    if MOUNT_DRIVE and any(k in os.environ for k in (\'COLAB_RELEASE_TAG\', \'COLAB_BACKEND_VERSION\')):\\n"\n            "        if not os.path.ismount(\'/content/drive\'):\\n"\n            "            from google.colab import drive\\n"\n            "            drive.mount(\'/content/drive\')\\n"\n            "        if not os.path.ismount(\'/content/drive\'): raise RuntimeError(\'Drive未实际挂载，停止以免误存到临时目录\')\\n"\n            "        ARCHIVE = Path(\'/content/drive/MyDrive/robot-lab/experiments/smolvla_peft_20261103_001\')\\n"\n            "        ARCHIVE.mkdir(parents=True, exist_ok=True)\\n"\n            "        LOG_ROOT = ARCHIVE / \'setup_logs\'\\n"\n            "    command = [sys.executable, str(SCRIPT), \'prepare\', \'--profile\', PROFILE, \'--workdir\', str(WORKDIR)]\\n"\n            "    if PROFILE == \'gpu\': command.append(\'--allow-managed-torch\')\\n"\n            "    command += [\'--log-root\', str(LOG_ROOT)]\\n"\n            "    if PREPARE_ENV:\\n"\n            "        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors=\'replace\') as process:\\n"\n            "            for line in process.stdout: print(line, end=\'\', flush=True)\\n"\n            "            code = process.wait()\\n"\n            "        if code: raise RuntimeError(f\'准备失败，退出码{code}；完整日志：{LOG_ROOT}\')\\n"\n            "        print(\'准备完成；不会自动训练。\')\\n"\n            "    else:\\n"\n            "        assert (WORKDIR / \'venv/bin/python\').is_file() and (WORKDIR / \'environment.json\').is_file(), \'本运行时尚未准备，需设PREPARE_ENV=True\'\\n"\n            "        print(\'仅同步代码，跳过安装/下载；未重新验证环境。后续显式选择评测入口。\')\\n"\n            "except Exception:\\n"\n            "    (WORKDIR / \'prepare_cell_error.log\').write_text(traceback.format_exc())\\n"\n            "    if ARCHIVE is not None: (ARCHIVE / \'prepare_cell_error.log\').write_text(traceback.format_exc())\\n"\n            "    raise\\n"\n            "finally:\\n"\n            "    if PREPARE_ENV and ARCHIVE is not None and (WORKDIR / \'logs\').is_dir():\\n"\n            "        saved_logs = ARCHIVE / (\'setup_logs_\' + datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S\'))\\n"\n            "        shutil.copytree(WORKDIR / \'logs\', saved_logs, dirs_exist_ok=False)\\n"\n            "        print(\'准备日志已归档到：\', saved_logs)\\n")\n    notebook = {"cells": [{"cell_type": "code", "id": "training-prepare", "metadata": {},\n                            "source": cell.splitlines(keepends=True), "outputs": [], "execution_count": None}],\n                "metadata": {"kernelspec": {"name": "python3", "display_name": "Python 3", "language": "python"}},\n                "nbformat": 4, "nbformat_minor": 5}\n    if args.notebook.exists():\n        existing = json.loads(args.notebook.read_text())\n        matches = [i for i, c in enumerate(existing[\'cells\']) if c.get(\'id\') == \'training-prepare\']\n        if len(matches) != 1:\n            raise RuntimeError("无法唯一定位自动生成的准备单元格；保留原Notebook，另选--notebook路径。")\n        previous_cell = existing[\'cells\'][matches[0]]\n        notebook[\'cells\'][0][\'outputs\'] = previous_cell.get(\'outputs\', [])\n        notebook[\'cells\'][0][\'execution_count\'] = previous_cell.get(\'execution_count\')\n        notebook[\'cells\'][0][\'metadata\'][\'source_updated_after_execution\'] = True\n        existing[\'cells\'][matches[0]] = notebook[\'cells\'][0]\n        notebook = existing  # 保留用户自己添加的GPU检查等单元格与输出。\n    write_json(args.notebook, notebook)\n    print(args.notebook)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("mode", choices=["prepare", "assets", "check-data", "cpu", "train", "export-eval", "score-eval", "check-metric", "prepare-sim", "repair-sim", "check-sim", "eval-libero", "export-notebook"])\n    parser.add_argument("--workdir", type=Path, default=Path("outputs/training_workbench"))\n    parser.add_argument("--profile", choices=["cpu", "gpu"], default="cpu")\n    parser.add_argument("--skip-assets", action="store_true")\n    parser.add_argument("--allow-managed-torch", action="store_true",\n                        help="GPU宿主Torch不兼容时在独立环境安装官方2.11.0/cu128，不修改宿主")\n    parser.add_argument("--archive-dir", type=Path)\n    parser.add_argument("--inspect-only", action="store_true", help="train入口只保存首个真实batch，不执行参数更新")\n    parser.add_argument("--practice-step-file", type=Path, help="train入口显式调用此文件的train_one_batch一次，然后结束；不影响正式五步入口")\n    metric_default = Path(__file__).with_name("evaluation_metrics.py")\n    if not metric_default.exists():\n        metric_default = Path(__file__).resolve().parents[1] / "studies/policy_generalization/evaluation_metrics.py"\n    parser.add_argument("--metric-file", type=Path, default=metric_default)\n    parser.add_argument("--checkpoint", type=Path, help="已训练checkpoint的pretrained_model目录")\n    parser.add_argument("--predictions", type=Path, help="score-eval读取的paired_predictions.pt")\n    parser.add_argument("--eval-run", default="heldout_825_001")\n    parser.add_argument("--eval-noise-seed", type=int, default=71)\n    parser.add_argument("--episodes", type=int, default=1, help="LIBERO每条件回合数，首次只跑1回合")\n    parser.add_argument("--seed", type=int, default=1000)\n    parser.add_argument("--sim-fps", type=int, default=20, help="明确的仿真控制/录像FPS，不从数据FPS推断")\n    parser.add_argument("--max-steps", type=int, default=280)\n    parser.add_argument("--condition", choices=["both", "base", "trained"], default="both")\n    parser.add_argument("--log-root", type=Path, help="可指定持久化日志目录；不影响权重/数据缓存位置")\n    parser.add_argument("--notebook", type=Path, default=Path("scripts/smolvla_training_prepare.ipynb"))\n    args = parser.parse_args(argv)\n    if min(args.episodes, args.sim_fps, args.max_steps) <= 0:\n        parser.error("episodes/sim-fps/max-steps必须为正整数")\n    if args.practice_step_file and (args.mode != "train" or args.inspect_only):\n        parser.error("--practice-step-file只用于train，不能与--inspect-only同时使用")\n    if args.mode == "export-notebook":\n        export_notebook(args)\n        return\n    args.workdir = args.workdir.absolute()\n    args.workdir.mkdir(parents=True, exist_ok=True)\n    check_persistent_path(args.archive_dir)\n    check_persistent_path(args.log_root)\n    os.environ.update(runtime_env(args.workdir))\n    os.environ.pop("PYTHONPATH", None)\n    if args.mode in ("train", "export-eval", "eval-libero"):\n        os.environ["HF_HUB_OFFLINE"] = "1"\n    log = Log((args.log_root or args.workdir / "logs") / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ") + "_" + args.mode))\n    original_stdout, original_stderr = sys.stdout, sys.stderr\n    console = (log.directory / "console.log").open("a", encoding="utf-8")\n    sys.stdout, sys.stderr = Tee(original_stdout, console), Tee(original_stderr, console)\n    try:\n        if args.mode in ("prepare-sim", "repair-sim", "check-sim", "eval-libero"):\n            import smolvla_libero_tools\n            function = {"prepare-sim": smolvla_libero_tools.prepare_entry,\n                        "repair-sim": smolvla_libero_tools.repair_entry,\n                        "check-sim": smolvla_libero_tools.check_entry,\n                        "eval-libero": smolvla_libero_tools.evaluate_entry}[args.mode]\n            function(args, log, sys.modules[__name__])\n        elif args.mode in ("export-eval", "score-eval", "check-metric"):\n            import smolvla_eval_tools\n            function = {"export-eval": smolvla_eval_tools.export_entry,\n                        "score-eval": smolvla_eval_tools.score_entry,\n                        "check-metric": smolvla_eval_tools.check_metric_entry}[args.mode]\n            function(args, log, sys.modules[__name__])\n        else:\n            {"prepare": prepare, "assets": assets, "check-data": check_data, "cpu": cpu_lab, "train": train}[args.mode](args, log)\n    except Exception as error:\n        log.say(traceback.format_exc())\n        write_json(log.directory / "status.json", {"ok": False, "type": type(error).__name__, "error": redact(str(error))})\n        raise SystemExit(1) from None\n    finally:\n        sys.stdout, sys.stderr = original_stdout, original_stderr\n        console.close()\n    write_json(log.directory / "status.json", {"ok": True, "mode": args.mode, "inspect_only": args.inspect_only})\n\n\nif __name__ == "__main__":\n    # 终端/ROS可能通过PYTHONPATH混入另一份库。只重启本CLI子进程，不改父终端或Notebook环境。\n    if "PYTHONPATH" in os.environ:\n        clean_env = os.environ.copy()\n        clean_env.pop("PYTHONPATH", None)\n        os.execve(sys.executable, [sys.executable, str(Path(__file__).resolve()), *sys.argv[1:]], clean_env)\n    main()\n'
SCRIPT = WORKDIR / 'smolvla_training_workbench.py'
SCRIPT.write_text(SOURCE)
PRACTICE_SOURCE = '"""你的单batch训练练习；由workbench的--practice-step-file显式调用。\n\n输入：已加载的LoRA策略、已经预处理并移到正确设备的batch、已配置的optimizer。\nSmolVLA的policy(batch)返回(loss, loss_dict)。返回Python float形式的loss。\n这里不再加载模型、不重复归一化、不调用其他训练器。\n"""\n\n\ndef train_one_batch(policy, batch, optimizer):\n    # TODO 1：设置训练模式。\n    policy.train()\n    # TODO 2：清理上一步梯度。\n    optimizer.zero_grad()\n    # TODO 3：前向并取得loss。\n    loss, loss_dict = policy(batch)\n    # TODO 4：反向传播。\n    loss.backward()\n    # TODO 5：更新参数。\n    optimizer.step()\n    # TODO 6：返回与计算图分离的数值loss。\n    return loss.detach().item()\n'
PRACTICE_FILE = WORKDIR / 'train_one_batch.py'
if not PRACTICE_FILE.exists(): PRACTICE_FILE.write_text(PRACTICE_SOURCE)
else: print('保留已有练习文件：', PRACTICE_FILE, '；本地修改后需显式同步此文件。')
METRICS_SOURCE = '"""用户实现的离线动作指标；由check-metric/score-eval显式调用。"""\n\nimport torch\n\n\ndef masked_action_mae(pred, target, is_pad):\n    """返回有效动作元素上的平均绝对误差，类型为Python float。\n\n    pred/target: 同形状浮点Tensor [B,H,D]，本次真实D=7。\n    is_pad: bool Tensor [B,H]；True代表补齐位置，必须排除。\n    按所有有效动作元素求平均，不能先对各样本平均再等权平均。\n    全部为padding、形状不匹配或mask非bool时，抛出ValueError/TypeError。\n    本次比较标准化动作，不在此函数中反归一化或加载模型。\n    """\n    # 1：检查输入形状、mask类型，以及是否存在有效位置。\n    if not all(isinstance(value, torch.Tensor) for value in (pred, target, is_pad)):\n        raise TypeError("pred/target/is_pad must be torch.Tensor")\n    if pred.ndim != 3 or target.ndim != 3 or is_pad.ndim != 2:\n        raise ValueError(f"pred/target/is_pad must be (B,H,D) or (B,H), got {tuple(pred.shape)}/{tuple(target.shape)}/{tuple(is_pad.shape)}")\n    if pred.shape != target.shape:\n        raise ValueError(f"pred/target must have same shape, got {tuple(pred.shape)}/{tuple(target.shape)}")\n    if is_pad.shape != pred.shape[:2]:\n        raise ValueError(f"is_pad must match (B,H)={tuple(pred.shape[:2])}, got {tuple(is_pad.shape)}")\n    if pred.numel() == 0:\n        raise ValueError("pred/target must contain action elements")\n    if not pred.is_floating_point() or not target.is_floating_point():\n        raise TypeError("pred/target must be floating-point tensors")\n    if pred.device != target.device or pred.device != is_pad.device:\n        raise ValueError("pred/target/is_pad must be on the same device")\n    if is_pad.dtype != torch.bool:\n        raise TypeError(f"is_pad must be bool, got {is_pad.dtype}")\n    if is_pad.all():\n        raise ValueError("is_pad must have at least one False value")\n    # 2：计算绝对误差，并排除补齐位置。保留用户实现。\n    valid_mask = ~is_pad  # True for valid positions\n    abs_error = torch.abs(pred - target)  # (B,H,D)\n    valid_abs_error = abs_error[valid_mask.unsqueeze(-1).expand_as(abs_error)]\n    if not torch.isfinite(valid_abs_error).all():\n        raise ValueError("valid action errors contain NaN or Inf")\n    # 3：对有效动作元素求平均，返回Python float。\n    mean_abs_error = valid_abs_error.mean().item()\n    return mean_abs_error\n'
EVAL_SOURCE = '"""评测接入代码；GPU只导出预测，用户指标在CPU单独计算。"""\nfrom datetime import datetime, timezone\nimport importlib.util\nimport json\nimport math\nfrom pathlib import Path\nimport shutil\n\n\ndef load_metric(path):\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(f"未找到用户指标文件：{path}")\n    spec = importlib.util.spec_from_file_location("user_evaluation_metrics", path)\n    module = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(module)\n    function = module.masked_action_mae\n    if not callable(function):\n        raise TypeError("指标文件须定义masked_action_mae。")\n    return function\n\n\ndef check_metric(function):\n    """小张量只核对接口与padding，不需要模型、CUDA或网络。"""\n    import torch\n    cases = [\n        (torch.tensor([[[1., 3.], [5., 7.]]]), torch.zeros(1, 2, dtype=torch.bool), 4.),\n        (torch.tensor([[[1., 3.], [999., 999.]]]), torch.tensor([[False, True]]), 2.),\n        (torch.tensor([[[1., 1.], [999., 999.]], [[3., 3.], [5., 5.]]]),\n         torch.tensor([[False, True], [False, False]]), 3.),\n    ]\n    for pred, mask, expected in cases:\n        result = function(pred, torch.zeros_like(pred), mask)\n        if not isinstance(result, float) or not math.isfinite(result) or not math.isclose(result, expected, abs_tol=1e-6):\n            raise AssertionError(f"指标结果应为float {expected}，实际为{result!r}。")\n    bad_inputs = [\n        (torch.zeros(1, 2, 2), torch.zeros(1, 2, 2), torch.ones(1, 2, dtype=torch.bool)),\n        (torch.zeros(1, 2, 2), torch.zeros(1, 3, 2), torch.zeros(1, 2, dtype=torch.bool)),\n        (torch.zeros(1, 2, 2), torch.zeros(1, 2, 2), torch.zeros(1, 2)),\n    ]\n    for args in bad_inputs:\n        try:\n            function(*args)\n        except (TypeError, ValueError):\n            continue\n        raise AssertionError("全padding、形状不匹配或非bool mask必须明确拒绝。")\n\n\ndef check_metric_entry(args, log, wb):\n    check_metric(load_metric(args.metric_file))\n    log.say("masked_action_mae的CPU接口与padding检查通过；尚未评测模型。")\n\n\ndef score_entry(args, log, wb):\n    import torch\n    function = load_metric(args.metric_file)\n    check_metric(function)\n    if not args.predictions:\n        raise ValueError("score-eval需要--predictions指向paired_predictions.pt。")\n    source = Path(args.predictions)\n    meta = json.loads((source.parent / "evaluation_manifest.json").read_text())\n    if not meta.get("complete"):\n        raise RuntimeError("预测导出未完成，保留现场，不能报告完整评测。")\n    if wb.digest(source) != meta.get("prediction_sha256"):\n        raise RuntimeError("预测文件的SHA256与导出记录不一致。")\n    packed = torch.load(source, map_location="cpu", weights_only=True)\n    target, mask = packed["target"], packed["is_pad"]\n    if mask.dtype != torch.bool or target.ndim != 3 or mask.shape != target.shape[:2]:\n        raise ValueError("预测文件的标签/mask格式不正确。")\n    if len(meta["samples"]) != len(target) or not torch.isfinite(target).all():\n        raise ValueError("样本记录/标签不完整或含非有限数。")\n    result = {"scope": "normalized action MAE; not closed-loop task success",\n              "prediction_sha256": wb.digest(source), "metric_sha256": wb.digest(args.metric_file),\n              "samples": meta["samples"], "valid_timesteps": int((~mask).sum()), "per_sample": []}\n    for key in ["base", "trained"]:\n        pred = packed[key]\n        if pred.shape != target.shape or not torch.isfinite(pred).all():\n            raise ValueError("预测形状错误或存在非有限数。")\n        result[key + "_mae"] = function(pred, target, mask)\n    result["trained_minus_base_mae"] = result["trained_mae"] - result["base_mae"]\n    result["prediction_change_mae"] = function(packed["trained"], packed["base"], mask)\n    for i, sample in enumerate(meta["samples"]):\n        result["per_sample"].append({**sample, **{key + "_mae": function(packed[key][i:i+1], target[i:i+1], mask[i:i+1]) for key in ["base", "trained"]}})\n    destination = source.parent / ("score_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ") + ".json")\n    wb.write_json(destination, result)\n    shutil.copy2(args.metric_file, destination.with_suffix(".py"))\n    log.say(result)\n    log.say(f"CPU评测已保存：{destination}；负的trained_minus_base_mae表示本次误差降低。")\n\n\ndef predict_pair(policy, batch, noise):\n    """同一份输入/初始噪声，比较冻结base与训练后adapter；不使用动作队列。"""\n    import torch\n    def copied():\n        return {k: v.clone() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}\n    with torch.inference_mode():\n        policy.reset()\n        with policy.disable_adapter():\n            base = policy.predict_action_chunk(copied(), noise=noise.clone()).detach().cpu()\n        policy.reset()\n        trained = policy.predict_action_chunk(copied(), noise=noise.clone()).detach().cpu()\n    return base, trained\n\n\ndef export_entry(args, log, wb):\n    import torch\n    from unittest.mock import patch\n    from torch.utils.data import default_collate\n    from lerobot.configs.policies import PreTrainedConfig\n    from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata\n    from lerobot.datasets.factory import resolve_delta_timestamps\n    from lerobot.datasets.lerobot_dataset import LeRobotDataset\n    from lerobot.policies.factory import make_policy, make_pre_post_processors\n    from transformers import AutoModelForImageTextToText\n    from peft import PeftConfig\n    import inspect\n\n    if not Path(inspect.getfile(make_policy)).resolve().is_relative_to((args.workdir / "lerobot").resolve()):\n        raise RuntimeError("评测导入了另一份LeRobot，请使用准备生成的解释器。")\n\n    manifest = json.loads((args.workdir / "environment.json").read_text())\n    if not manifest.get("assets_ready"):\n        raise RuntimeError("先恢复准备好的模型/数据环境；本入口不下载资产或启动训练。")\n    candidates = [args.workdir / "train_run/checkpoints/000005/pretrained_model"]\n    if args.archive_dir:\n        candidates.append(args.archive_dir / "checkpoints/000005/pretrained_model")\n    checkpoint = Path(args.checkpoint) if args.checkpoint else next((p for p in candidates if p.is_dir()), None)\n    if checkpoint is None or not (checkpoint / "adapter_config.json").is_file():\n        raise FileNotFoundError("没有找到训练后的adapter checkpoint；用--checkpoint指定pretrained_model目录。")\n    evidence_path = checkpoint.parent / "training_evidence.json"\n    evidence = json.loads(evidence_path.read_text())\n    if not evidence.get("trainable") or any("lora_" not in key for key in evidence["trainable"]):\n        raise RuntimeError("此对照要求只训练LoRA；检测到其他可训练参数，不能把禁用adapter当作训练前base。")\n    adapter = PeftConfig.from_pretrained(checkpoint)\n    if adapter.base_model_name_or_path != wb.BASE_ID or adapter.revision != wb.BASE_SHA or adapter.modules_to_save or adapter.bias != "none":\n        raise RuntimeError("checkpoint的base版本或训练方式不符合本次冻结base对照。")\n    if not args.eval_run or Path(args.eval_run).name != args.eval_run or args.eval_run in (".", ".."):\n        raise ValueError("--eval-run必须是一个目录名。")\n    output = (args.archive_dir or args.workdir) / "evaluations" / args.eval_run\n    if output.exists():\n        raise FileExistsError(f"保留已有评测：{output}；请更换--eval-run。")\n    cfg = PreTrainedConfig.from_pretrained(checkpoint)\n    if not cfg.use_peft:\n        raise RuntimeError("checkpoint配置没有启用PEFT，不能执行LoRA启用/禁用对照。")\n    cfg.pretrained_path, cfg.device = checkpoint, "cuda"\n    episode = 825\n    if episode in wb.TRAIN_EPISODES:\n        raise RuntimeError("验证episode不能与训练episode重叠。")\n    wb.validate_dataset_files(Path(manifest["dataset_root"]))\n    ds_meta = LeRobotDatasetMetadata(wb.DATA_ID, root=manifest["dataset_root"], revision=wb.DATA_SHA)\n    dataset = LeRobotDataset(wb.DATA_ID, root=manifest["dataset_root"], revision=wb.DATA_SHA, episodes=[episode],\n                            delta_timestamps=resolve_delta_timestamps(cfg, ds_meta), video_backend="pyav")\n    points = sorted({0, len(dataset) // 2, len(dataset) - 1})\n    if len(dataset) != 153:\n        raise RuntimeError("验证轨迹长度与固定数据版本不一致。")\n    preprocessor, _ = make_pre_post_processors(policy_cfg=cfg, pretrained_path=checkpoint,\n                                              preprocessor_overrides={"device_processor": {"device": "cpu"}})\n    cpu_batches = [preprocessor(default_collate([dataset[frame]])) for frame in points]\n    for batch in cpu_batches:\n        target, mask = batch["action"], batch["action_is_pad"]\n        if (target.shape != (1, cfg.chunk_size, 7) or mask.shape != target.shape[:2]\n                or mask.dtype != torch.bool or not (~mask).any() or not torch.isfinite(target).all()):\n            raise RuntimeError("CPU预处理得到的标签或padding不符合协议；尚未加载GPU模型。")\n    # 以下才开始使用GPU。所有文件/数据协议检查已完成，不需要用户指标已实现。\n    if not torch.cuda.is_available():\n        raise RuntimeError("导出预测需要GPU；check-metric/score-eval仅需CPU。")\n    original_loader = AutoModelForImageTextToText.from_pretrained\n    def load_vlm(*a, **kw):\n        if "torch_dtype" in kw:\n            kw.setdefault("dtype", kw.pop("torch_dtype"))\n        kw.setdefault("token", False)\n        return original_loader(*a, **kw)\n    output.mkdir(parents=True)\n    meta = {"complete": False, "scope": "paired offline prediction on heldout episode; no environment rollout",\n            "checkpoint": str(checkpoint), "adapter_sha256": wb.digest(checkpoint / "adapter_model.safetensors"),\n            "base_revision": wb.BASE_SHA, "dataset_revision": wb.DATA_SHA, "episode": episode,\n            "planned_frames": points, "noise_seed": args.eval_noise_seed, "samples": [],\n            "inference_config": {key: getattr(cfg, key) for key in\n                                 ["chunk_size", "num_steps", "max_action_dim", "max_state_dim", "use_cache"]},\n            "metric_required_for_export": False, "source_manifest": manifest}\n    wb.write_json(output / "evaluation_manifest.json", meta)\n    with patch.object(AutoModelForImageTextToText, "from_pretrained", staticmethod(load_vlm)):\n        policy = make_policy(cfg=cfg, ds_meta=ds_meta)\n    policy.eval()\n    policy.requires_grad_(False)\n    buckets = {key: [] for key in ["base", "trained", "target", "is_pad"]}\n    for frame, cpu_batch in zip(points, cpu_batches, strict=True):\n        batch = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in cpu_batch.items()}\n        target = batch["action"].detach().cpu()\n        mask = batch["action_is_pad"].detach().cpu()\n        noise = torch.randn(1, policy.config.chunk_size, policy.config.max_action_dim,\n                            generator=torch.Generator().manual_seed(args.eval_noise_seed)).to("cuda")\n        base, trained = predict_pair(policy, batch, noise)\n        if base.shape != target.shape or trained.shape != target.shape or not all(torch.isfinite(v).all() for v in [base, trained, target]):\n            raise RuntimeError("预测/标签形状不一致或出现非有限数，保留未完成输出。")\n        if mask.dtype != torch.bool or mask.shape != target.shape[:2] or not (~mask).any():\n            raise RuntimeError("动作padding mask不正确。")\n        if frame == points[0]:\n            repeated_base, repeated_trained = predict_pair(policy, batch, noise)\n            torch.testing.assert_close(repeated_base, base, rtol=1e-3, atol=1e-4)\n            torch.testing.assert_close(repeated_trained, trained, rtol=1e-3, atol=1e-4)\n            meta["repeatability_checked"] = True\n        for key, value in [("base", base), ("trained", trained), ("target", target), ("is_pad", mask)]:\n            buckets[key].append(value)\n        meta["samples"].append({"episode": episode, "frame": frame, "valid_timesteps": int((~mask).sum())})\n        wb.save_inspection_images(batch, output / f"frame_{frame:03d}")\n        temporary = output / "paired_predictions.pt.tmp"\n        torch.save({key: torch.cat(value, dim=0) for key, value in buckets.items()}, temporary)\n        temporary.replace(output / "paired_predictions.pt")\n        wb.write_json(output / "evaluation_manifest.json", meta)\n        log.say(meta["samples"][-1])\n    meta["complete"] = True\n    meta["prediction_sha256"] = wb.digest(output / "paired_predictions.pt")\n    wb.write_json(output / "evaluation_manifest.json", meta)\n    log.say(f"预测已保存：{output}。现在可释放GPU，masked_action_mae与score-eval可在本地CPU完成。")\n'
METRICS_FILE = WORKDIR / 'evaluation_metrics.py'
if not METRICS_FILE.exists(): METRICS_FILE.write_text(METRICS_SOURCE)
(WORKDIR / 'smolvla_eval_tools.py').write_text(EVAL_SOURCE)
SIM_SOURCE = '"""保存checkpoint的LIBERO闭环评测；复用官方rollout/处理器，不实现另一个控制循环。"""\nfrom contextlib import nullcontext\nimport inspect\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\n\nLIBERO_SHA = "8f1084e3132a39270c3a13ebe37270a43ece2a01"\nLIBERO_REPO = "https://github.com/Lifelong-Robot-Learning/LIBERO.git"\nTASK_LANGUAGE = "pick up the alphabet soup and place it in the basket"\nSIM_PINS = ["robosuite==1.4.1", "bddl==1.0.1", "easydict==1.13", "mujoco==3.7.0",\n            "future==1.0.0", "Wand==0.6.13", "scikit-image==0.25.2", "gym==0.26.2",\n            "matplotlib==3.10.8", "numba==0.61.2", "hydra-core>=1.2,<1.4", "h5py>=3.11,<4", "cloudpickle>=2"]\n\n\ndef sim_environment(workdir, wb):\n    env = wb.runtime_env(workdir)\n    env.update(MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",\n               LIBERO_CONFIG_PATH=str(workdir / "libero-config"),\n               MPLCONFIGDIR=str(workdir / "matplotlib-cache"), NUMBA_CACHE_DIR=str(workdir / "numba-cache"))\n    return env\n\n\ndef find_task(benchmark):\n    """按原始任务语言定位，拒绝view后缀；数据集task_index不是suite内部task_id。"""\n    suite = benchmark.get_benchmark_dict()["libero_object"]()\n    matches = [(i, task) for i, task in enumerate(suite.tasks)\n               if " ".join(task.language.lower().split()) == TASK_LANGUAGE]\n    if len(matches) != 1:\n        raise RuntimeError("未唯一找到原始alphabet soup任务；不能用带view后缀的Plus任务代替。")\n    return matches[0]\n\n\ndef register_source_root(site_dir, source):\n    """LIBERO外层是namespace目录，旧setup.py的find_packages可能漏掉它。"""\n    source = Path(source).resolve()\n    if not (source / "libero/libero/__init__.py").is_file():\n        raise FileNotFoundError(f"LIBERO源码布局不完整：{source}")\n    target = Path(site_dir) / "libero_source_root.pth"\n    target.write_text(str(source) + "\\n")\n    return target\n\n\ndef repair_entry(args, log, wb):\n    """已有依赖安装成功时，只修复导入路径并复查渲染，不安装、不加载VLA。"""\n    work = args.workdir\n    python = work / "sim-venv/bin/python"\n    source = work / "libero-original"\n    if not python.is_file() or not (work / "libero-config/config.yaml").is_file():\n        raise RuntimeError("没有已安装的仿真环境，请用prepare-sim完成首次准备。")\n    site = Path(subprocess.check_output([str(python), "-c",\n                "import sysconfig; print(sysconfig.get_path(\'purelib\'))"], text=True).strip())\n    env = sim_environment(work, wb)\n    register_source_root(site, source)\n    log.say("已在独立sim-venv注册LIBERO源码根目录；不修改第三方源码或训练venv。")\n    log.run([python, "-c", "import libero.libero as m; print(\'LIBERO实际导入文件:\', m.__file__)"], env=env)\n    version_check = ("from packaging.requirements import Requirement; from importlib.metadata import version; "\n                     f"items=[Requirement(s) for s in {SIM_PINS!r}]; "\n                     "bad=[str(r) for r in items if version(r.name) not in r.specifier]; "\n                     "assert not bad, \'依赖版本不符合本次协议，请运行prepare-sim: \'+str(bad); "\n                     "print(\'固定仿真依赖版本核对通过\')")\n    log.run([python, "-c", version_check], env=env)\n    log.run([python, "-m", "pip", "check"], env=env)\n    wb.write_json(work / "simulation.json", {"ready": False, "status": "repairing", "libero_revision": LIBERO_SHA})\n    log.run([python, Path(wb.__file__).resolve(), "check-sim", "--workdir", work,\n             "--log-root", log.directory / "render_probe", "--sim-fps", str(args.sim_fps)], env=env)\n    wb.write_json(work / "simulation.json", {"ready": True, "libero_revision": LIBERO_SHA,\n                  "python": str(python), "source": str(source), "requirements": SIM_PINS,\n                  "render_probe": str(log.directory / "render_probe"), "sim_fps": args.sim_fps,\n                  "repair": "namespace source-root registration; no package installation"})\n    log.say("路径修复与真实渲染复查完成；没有重新训练或加载VLA。")\n\n\ndef prepare_entry(args, log, wb):\n    """独立sim-venv借用训练依赖；不修改已跑通的训练venv或宿主Python。"""\n    work = args.workdir\n    training_python = work / "venv/bin/python"\n    if not training_python.is_file() or not (work / "environment.json").is_file():\n        raise RuntimeError("先运行训练环境prepare；仿真环境会借用其已验证的模型依赖。")\n    training = json.loads((work / "environment.json").read_text())\n    if training.get("profile") != "gpu" or training.get("lerobot_revision") != wb.LEROBOT_SHA:\n        raise RuntimeError("需要已验证的真实SmolVLA训练环境，不能使用CPU小模型练习目录。")\n    env = sim_environment(work, wb)\n    env.pop("HF_HUB_OFFLINE", None)\n    ready_path = work / "simulation.json"\n    previous = json.loads(ready_path.read_text()) if ready_path.is_file() else {}\n    if (previous.get("ready") and previous.get("requirements") == SIM_PINS\n            and previous.get("libero_revision") == LIBERO_SHA and (work / "sim-venv/bin/python").is_file()):\n        site = Path(subprocess.check_output([str(work / "sim-venv/bin/python"), "-c",\n                "import sysconfig; print(sysconfig.get_path(\'purelib\'))"], text=True).strip())\n        register_source_root(site, work / "libero-original")\n        previous["ready"] = False\n        wb.write_json(ready_path, previous)\n        log.run([work / "sim-venv/bin/python", "-m", "pip", "check"], env=env)\n        log.run([work / "sim-venv/bin/python", Path(wb.__file__).resolve(), "check-sim", "--workdir", work,\n                 "--log-root", log.directory / "render_probe", "--sim-fps", str(args.sim_fps)], env=env)\n        previous.update(ready=True, render_probe=str(log.directory / "render_probe"), sim_fps=args.sim_fps)\n        wb.write_json(ready_path, previous)\n        log.say("复用现有仿真环境；真实渲染/录像复查通过，没有重装依赖。")\n        return\n    if shutil.disk_usage(work).free < 3 * 2**30:\n        raise RuntimeError("首次仿真准备至少预留3GiB磁盘，保留已有文件，不清缓存。")\n    wb.write_json(ready_path, {"ready": False, "status": "preparing", "libero_revision": LIBERO_SHA})\n    sim_python = wb.ensure_environment(work / "sim-venv", log, env)\n    train_site = subprocess.check_output([str(training_python), "-c",\n                    "import sysconfig; print(sysconfig.get_path(\'purelib\'))"], text=True).strip()\n    sim_site = Path(subprocess.check_output([str(sim_python), "-c",\n                    "import sysconfig; print(sysconfig.get_path(\'purelib\'))"], text=True).strip())\n    # addsitedir也处理训练环境的editable .pth；sim自己安装的包排在前面。\n    (sim_site / "training_dependencies.pth").write_text(f"import site; site.addsitedir({train_site!r})\\n")\n    if sys.platform != "linux":\n        raise RuntimeError("此EGL准备入口面向Colab/AutoDL的Linux运行时。")\n    if os.geteuid() == 0:\n        log.run(["apt-get", "update", "-qq"], env=env)\n        log.run(["apt-get", "install", "-y", "libegl1", "libgl1", "libgles2", "libexpat1",\n                 "libfontconfig1", "libmagickwand-dev"], env=env)\n    else:\n        log.say("非root，跳过apt；系统须已安装EGL/OpenGL/ImageMagick运行库，后面的真实渲染检查会验证。")\n    source = work / "libero-original"\n    if not (source / ".git").is_dir():\n        if source.exists() and any(source.iterdir()):\n            raise RuntimeError(f"保留已有非Git目录：{source}")\n        source.mkdir(exist_ok=True)\n        log.run(["git", "init", str(source)], env=env)\n        log.run(["git", "-C", str(source), "remote", "add", "origin", LIBERO_REPO], env=env)\n    head = subprocess.run(["git", "-C", str(source), "rev-parse", "--verify", "HEAD"],\n                          capture_output=True, text=True)\n    if head.returncode:\n        # 首次fetch断网留下的空仓库可重试；不清空目录或重置已存在的提交。\n        log.run(["git", "-C", str(source), "fetch", "--depth=1", "origin", LIBERO_SHA], env=env)\n        log.run(["git", "-C", str(source), "checkout", "--detach", "FETCH_HEAD"], env=env)\n    revision = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()\n    if revision != LIBERO_SHA:\n        raise RuntimeError("已有LIBERO版本不同，保留现场，不自动切换。")\n    for required in ["libero/libero/assets/scenes", "libero/libero/assets/textures",\n                     "libero/libero/init_files/libero_object", "libero/libero/bddl_files/libero_object"]:\n        if not (source / required).is_dir():\n            raise RuntimeError(f"LIBERO源码/资产不完整，保留现场：{source / required}")\n    constraints = work / "simulation-constraints.txt"\n    torch_pin = subprocess.check_output([str(training_python), "-c",\n                      "from importlib.metadata import version; print(\'torch==\'+version(\'torch\'))"], text=True).strip()\n    constraints.write_text("\\n".join([torch_pin, *wb.PINS]) + "\\n")\n    # 不使用lerobot[libero]，不引入hf-libero/robomimic的EGL probe构建分支。\n    log.run([sim_python, "-m", "pip", "install", "--constraint", constraints,\n             "--report", log.directory / "pip-simulation.json", *SIM_PINS], env=env)\n    log.run([sim_python, "-m", "pip", "install", "--no-deps", "-e", source], env=env)\n    register_source_root(sim_site, source)\n    root = source / "libero/libero"\n    # import前写配置，避免子进程input()导致EOFError。\n    config = {"benchmark_root": str(root), "bddl_files": str(root / "bddl_files"),\n              "init_states": str(root / "init_files"), "assets": str(root / "assets"),\n              "datasets": str(work / "libero-unused-datasets")}\n    Path(config["datasets"]).mkdir(exist_ok=True)\n    config_file = work / "libero-config/config.yaml"\n    config_file.parent.mkdir(exist_ok=True)\n    # JSON是有效YAML，避免在宿主另装yaml。\n    config_file.write_text(json.dumps(config, indent=2) + "\\n")\n    macros = ("import importlib.util, pathlib, shutil; "\n              "p=pathlib.Path(importlib.util.find_spec(\'robosuite\').origin).parent; "\n              "q=p/\'macros_private.py\'; "\n              f"shutil.copyfile(p/\'macros.py\',q) if not q.exists() and p.is_relative_to(pathlib.Path({str(sim_site)!r})) else None")\n    log.run([sim_python, "-c", macros], env=env)\n    log.run([sim_python, "-c", "import libero.libero as m; print(\'LIBERO实际导入文件:\', m.__file__)"], env=env)\n    log.run([sim_python, "-m", "pip", "check"], env=env)\n    frozen = subprocess.check_output([str(sim_python), "-m", "pip", "freeze"], text=True, env=env)\n    (log.directory / "simulation-pip-freeze.txt").write_text(wb.redact(frozen))\n    # 不加载VLA；实际reset、图像预处理、MP4编码由独立进程检查。\n    log.run([sim_python, Path(wb.__file__).resolve(), "check-sim", "--workdir", work,\n             "--log-root", log.directory / "render_probe", "--sim-fps", str(args.sim_fps)], env=env)\n    wb.write_json(work / "simulation.json", {"ready": True, "libero_revision": LIBERO_SHA,\n                  "python": str(sim_python), "source": str(source), "config": config,\n                  "requirements": SIM_PINS,\n                  "render_probe": str(log.directory / "render_probe"), "sim_fps": args.sim_fps})\n    log.say("独立LIBERO环境及真实渲染检查完成；未加载GPU策略、未训练。")\n\n\ndef configure(args, wb):\n    os.environ.update(sim_environment(args.workdir, wb))\n    source = args.workdir / "libero-original"\n    if not (args.workdir / "libero-config/config.yaml").is_file():\n        raise RuntimeError("先执行prepare-sim，配置缺失时不导入LIBERO，避免交互提示。")\n    actual = subprocess.check_output(["git", "-C", str(source), "rev-parse", "HEAD"], text=True).strip()\n    if actual != LIBERO_SHA:\n        raise RuntimeError("实际LIBERO提交与固定协议不同。")\n    import libero.libero as libero\n    if not Path(libero.__file__).resolve().is_relative_to(source.resolve()):\n        raise RuntimeError("导入了其他LIBERO，请用sim-venv/bin/python执行。")\n    from libero.libero import benchmark\n    task_id, task = find_task(benchmark)\n    from lerobot.envs.configs import LiberoEnv\n    cfg = LiberoEnv(task="libero_object", task_ids=[task_id], fps=args.sim_fps,\n                    episode_length=args.max_steps, observation_height=256, observation_width=256,\n                    control_mode="relative", is_libero_plus=False,\n                    camera_name_mapping={"agentview_image": "image", "robot0_eye_in_hand_image": "image2"})\n    return cfg, task\n\n\ndef make_vector(cfg):\n    from lerobot.envs.factory import make_env\n    env = make_env(cfg, n_envs=1, use_async_envs=False)[cfg.task][cfg.task_ids[0]]\n    return configure_vector(env, cfg.fps, cfg.episode_length)\n\n\ndef configure_vector(env, fps, horizon):\n    """实例级录像FPS；TimeLimit让超时也产生done，避免官方聚合器argmax误判。"""\n    import gymnasium as gym\n    env.metadata = {**env.metadata, "render_fps": fps}\n    for index, subenv in enumerate(env.envs):\n        subenv.metadata = {**subenv.metadata, "render_fps": fps}\n        env.envs[index] = gym.wrappers.TimeLimit(subenv, max_episode_steps=horizon)\n    return env\n\n\ndef verify_video(path, fps):\n    import av\n    path = Path(path)\n    if not path.is_file() or path.stat().st_size == 0:\n        raise RuntimeError(f"录像未生成：{path}")\n    with av.open(str(path)) as container:\n        stream = container.streams.video[0]\n        rate = float(stream.average_rate)\n        count = sum(1 for _ in container.decode(video=0))\n    if count < 1 or abs(rate - fps) > 1e-6:\n        raise RuntimeError(f"录像帧数/播放FPS异常：{count}/{rate}")\n    return {"path": str(path), "frames": count, "playback_fps": rate, "duration_s": count / rate}\n\n\ndef check_entry(args, log, wb):\n    import numpy as np\n    import torch\n    from lerobot.envs.utils import preprocess_observation\n    from lerobot.utils.io_utils import write_video\n    from lerobot.utils.random_utils import set_seed\n    cfg, task = configure(args, wb)\n    env = make_vector(cfg)\n    try:\n        set_seed(args.seed)\n        raw, _ = env.reset(seed=[args.seed])\n        batch = preprocess_observation(raw)\n        batch["task"] = [task.language]\n        env_pre, _ = cfg.get_env_processors()\n        batch = env_pre(batch)\n        if batch["observation.state"].shape != (1, 8):\n            raise RuntimeError("环境状态不是[eef_pos3,axis_angle3,gripper_qpos2]。")\n        for key in ["observation.images.image", "observation.images.image2"]:\n            if batch[key].shape != (1, 3, 256, 256) or not torch.isfinite(batch[key]).all():\n                raise RuntimeError(f"相机输入错误：{key}")\n        wb.save_inspection_images(batch, log.directory)\n        frame = env.call("render")[0]\n        initial = torch.as_tensor(env.envs[0].unwrapped._env.get_sim_state()).clone()\n        from lerobot.envs.libero import get_libero_dummy_action\n        _, _, _, _, step_info = env.step(np.asarray([get_libero_dummy_action()], dtype=np.float64))\n        if "is_success" not in step_info:\n            raise RuntimeError("环境step没有提供is_success，不能统计任务成功率。")\n        video = log.directory / "render_probe.mp4"\n        write_video(video, [frame, env.call("render")[0]], args.sim_fps)\n        probe = verify_video(video, args.sim_fps)\n    finally:\n        env.close()\n    repeated_env = make_vector(cfg)\n    try:\n        set_seed(args.seed)\n        repeated_env.reset(seed=[args.seed])\n        repeated = torch.as_tensor(repeated_env.envs[0].unwrapped._env.get_sim_state())\n        torch.testing.assert_close(repeated, initial, rtol=1e-5, atol=1e-6)\n    finally:\n        repeated_env.close()\n    wb.write_json(log.directory / "render_check.json", {"ok": True, "task": task.language,\n                  "state_shape": list(batch["observation.state"].shape), "video": probe,\n                  "step_success_available": True, "paired_initial_state_checked": True})\n    log.say(f"真实场景reset/step、初始状态配对、两路图像/状态、MP4编码通过：{task.language}")\n\n\ndef evaluate_entry(args, log, wb):\n    """核心链：环境观测→环境处理→保存的processor→select_action→反归一化→env.step。"""\n    import torch\n    from unittest.mock import patch\n    from transformers import AutoModelForImageTextToText\n    from lerobot.configs.policies import PreTrainedConfig\n    from lerobot.datasets.dataset_metadata import LeRobotDatasetMetadata\n    from lerobot.policies.factory import make_policy, make_pre_post_processors\n    from lerobot.scripts.lerobot_eval import eval_policy\n    from lerobot.utils.random_utils import set_seed\n    from peft import PeftConfig\n\n    sim = json.loads((args.workdir / "simulation.json").read_text())\n    if not sim.get("ready") or sim["libero_revision"] != LIBERO_SHA:\n        raise RuntimeError("仿真准备/真实渲染检查尚未完成。")\n    if not Path(inspect.getfile(make_policy)).resolve().is_relative_to((args.workdir / "lerobot").resolve()):\n        raise RuntimeError("LeRobot源码导入路径与准备环境不同。")\n    env_cfg, task = configure(args, wb)\n    manifest = json.loads((args.workdir / "environment.json").read_text())\n    candidates = [args.workdir / "train_run/checkpoints/000005/pretrained_model"]\n    if args.archive_dir:\n        candidates.append(args.archive_dir / "checkpoints/000005/pretrained_model")\n    checkpoint = args.checkpoint or next((p for p in candidates if p.is_dir()), None)\n    if checkpoint is None:\n        raise FileNotFoundError("找不到官方五步checkpoint，请用--checkpoint指定pretrained_model。")\n    adapter = PeftConfig.from_pretrained(checkpoint)\n    evidence = json.loads((checkpoint.parent / "training_evidence.json").read_text())\n    if (adapter.base_model_name_or_path != wb.BASE_ID or adapter.revision != wb.BASE_SHA\n            or adapter.modules_to_save or adapter.bias != "none" or not evidence.get("trainable")\n            or any("lora_" not in n for n in evidence["trainable"])):\n        raise RuntimeError("此配对评测要求固定base、仅训练LoRA，不符合条件。")\n    ds_meta = LeRobotDatasetMetadata(wb.DATA_ID, root=manifest["dataset_root"], revision=wb.DATA_SHA)\n    languages = [str(name) for name, row in ds_meta.tasks.iterrows() if int(row["task_index"]) == 24]\n    if languages != [TASK_LANGUAGE]:\n        raise RuntimeError("数据任务24与仿真目标任务不一致。")\n    cfg = PreTrainedConfig.from_pretrained(checkpoint)\n    cfg.pretrained_path, cfg.device = checkpoint, "cuda"\n    if set(cfg.image_features) != {"observation.images.image", "observation.images.image2"}:\n        raise RuntimeError("本协议要求保存策略使用image/image2两路相机。")\n    if not cfg.use_peft or cfg.input_features["observation.state"].shape != (8,) or cfg.output_features["action"].shape != (7,):\n        raise RuntimeError("保存策略的状态/动作维度或PEFT配置不符。")\n    if Path(args.eval_run).name != args.eval_run or args.eval_run in ("", ".", ".."):\n        raise ValueError("eval-run必须是单个目录名。")\n    output = (args.archive_dir or args.workdir) / "libero_evaluations" / args.eval_run\n    if output.exists():\n        raise FileExistsError(f"保留已有仿真结果：{output}，请更换eval-run。")\n    if not torch.cuda.is_available():\n        raise RuntimeError("完整VLA闭环请在T4运行；本地只做指标/接入检查。")\n    # CPU和EGL检查应在prepare-sim中完成；这里复用成功的G2重载路径。\n    original = AutoModelForImageTextToText.from_pretrained\n    def load_vlm(*a, **kw):\n        if "torch_dtype" in kw:\n            kw.setdefault("dtype", kw.pop("torch_dtype"))\n        kw.setdefault("token", False)\n        return original(*a, **kw)\n    output.mkdir(parents=True)\n    protocol = {"complete": False, "scope": "single-task closed-loop protocol smoke; not generalization benchmark",\n                "task": task.language, "suite": env_cfg.task, "task_id": env_cfg.task_ids[0],\n                "checkpoint": str(checkpoint), "adapter_sha256": wb.digest(checkpoint / "adapter_model.safetensors"),\n                "libero_revision": LIBERO_SHA, "lerobot_revision": wb.LEROBOT_SHA, "base_revision": wb.BASE_SHA,\n                "dataset_revision": wb.DATA_SHA, "training_data_fps": ds_meta.fps,\n                "control_fps": args.sim_fps, "playback_fps": args.sim_fps, "control_mode": "relative",\n                "frequency_note": "data FPS does not prove simulator control rate; this explicit protocol needs behavior verification",\n                "max_steps": args.max_steps, "episodes_per_condition": args.episodes, "seed": args.seed,\n                "conditions": args.condition, "success_rate_units": "fraction in [0,1]",\n                "n_action_steps": cfg.n_action_steps, "flow_steps": cfg.num_steps, "results": []}\n    wb.write_json(output / "eval_info.json", protocol)\n    with patch.object(AutoModelForImageTextToText, "from_pretrained", staticmethod(load_vlm)):\n        policy = make_policy(cfg=cfg, ds_meta=ds_meta)\n    policy.eval()\n    policy.requires_grad_(False)\n    pre, post = make_pre_post_processors(policy_cfg=cfg, pretrained_path=checkpoint,\n                  preprocessor_overrides={"device_processor": {"device": "cuda"}})\n    env_pre, env_post = env_cfg.get_env_processors()\n    conditions = ["base", "trained"] if args.condition == "both" else [args.condition]\n    initial_states = {}\n    for condition in conditions:\n        env = make_vector(env_cfg)  # 新建环境使两个条件从同一初始状态序列起步\n        try:\n            for episode in range(args.episodes):\n                set_seed(args.seed + episode)\n                destination = output / condition / f"episode_{episode:03d}"\n                destination.mkdir(parents=True)\n                first = []\n                control_steps = 0\n                def observed_preprocessor(batch):\n                    nonlocal control_steps\n                    control_steps += 1\n                    if not first:\n                        # 环境processor已翻转图像/拼接8维状态；这里不能再翻转一次。\n                        state = torch.as_tensor(env.envs[0].unwrapped._env.get_sim_state()).clone()\n                        first.append(state)\n                        key = episode\n                        if condition == "trained" and key in initial_states:\n                            torch.testing.assert_close(state, initial_states[key], rtol=1e-5, atol=1e-6)\n                        else:\n                            initial_states[key] = state\n                        torch.save(state, destination / "initial_sim_state.pt")\n                        wb.save_inspection_images(batch, destination / "input_images")\n                    processed = pre(batch)\n                    if not (destination / "first_policy_input.pt").exists():\n                        torch.save({k: v.detach().cpu() for k, v in processed.items()\n                                    if isinstance(v, torch.Tensor)}, destination / "first_policy_input.pt")\n                    return processed\n                with policy.disable_adapter() if condition == "base" else nullcontext():\n                    info = eval_policy(env=env, policy=policy, env_preprocessor=env_pre,\n                         env_postprocessor=env_post, preprocessor=observed_preprocessor, postprocessor=post,\n                         n_episodes=1, max_episodes_rendered=1, videos_dir=destination / "videos",\n                         start_seed=args.seed + episode)\n                videos = [verify_video(p, args.sim_fps) for p in info.get("video_paths", [])]\n                if len(videos) != 1:\n                    raise RuntimeError("本回合没有生成完整视频，不记作已完成评测。")\n                if videos[0]["frames"] != control_steps:\n                    raise RuntimeError("录像帧数与实际控制步数不一致，保留现场，不记作完整回合。")\n                from PIL import Image\n                Image.fromarray(env.call("render")[0]).save(destination / "final_frame.png")\n                raw_env = env.envs[0].unwrapped\n                initial_index = (raw_env.init_state_id - 1) % len(raw_env._init_states)\n                row = {"condition": condition, "episode": episode, "initial_state_index": initial_index,\n                       "control_steps": control_steps,\n                       "seed": args.seed + episode, "success": bool(info["per_episode"][0]["success"]),\n                       "metrics": info["aggregated"], "videos": videos}\n                wb.write_json(destination / "episode_result.json", row)\n                protocol["results"].append(row)\n                wb.write_json(output / "eval_info.json", protocol)  # 每回合立即归档；断连仍有部分证据\n                log.say(row)\n        finally:\n            env.close()\n    protocol["success_rates"] = {condition: sum(r["success"] for r in protocol["results"] if r["condition"] == condition)\n                                / args.episodes for condition in conditions}\n    protocol["success_counts"] = {condition: sum(r["success"] for r in protocol["results"] if r["condition"] == condition)\n                                 for condition in conditions}\n    protocol["complete"] = True\n    wb.write_json(output / "eval_info.json", protocol)\n    log.say(f"LIBERO执行和录像完成：{output}；success_rates={protocol[\'success_rates\']}，单任务少量回合不能代表泛化。")\n'
(WORKDIR / 'smolvla_libero_tools.py').write_text(SIM_SOURCE)
ARCHIVE = None
LOG_ROOT = WORKDIR / 'logs'
try:
    if MOUNT_DRIVE and any(k in os.environ for k in ('COLAB_RELEASE_TAG', 'COLAB_BACKEND_VERSION')):
        if not os.path.ismount('/content/drive'):
            from google.colab import drive
            drive.mount('/content/drive')
        if not os.path.ismount('/content/drive'): raise RuntimeError('Drive未实际挂载，停止以免误存到临时目录')
        ARCHIVE = Path('/content/drive/MyDrive/robot-lab/experiments/smolvla_peft_20261103_001')
        ARCHIVE.mkdir(parents=True, exist_ok=True)
        LOG_ROOT = ARCHIVE / 'setup_logs'
    command = [sys.executable, str(SCRIPT), 'prepare', '--profile', PROFILE, '--workdir', str(WORKDIR)]
    if PROFILE == 'gpu': command.append('--allow-managed-torch')
    command += ['--log-root', str(LOG_ROOT)]
    if PREPARE_ENV:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace') as process:
            for line in process.stdout: print(line, end='', flush=True)
            code = process.wait()
        if code: raise RuntimeError(f'准备失败，退出码{code}；完整日志：{LOG_ROOT}')
        print('准备完成；不会自动训练。')
    else:
        assert (WORKDIR / 'venv/bin/python').is_file() and (WORKDIR / 'environment.json').is_file(), '本运行时尚未准备，需设PREPARE_ENV=True'
        print('仅同步代码，跳过安装/下载；未重新验证环境。后续显式选择评测入口。')
except Exception:
    (WORKDIR / 'prepare_cell_error.log').write_text(traceback.format_exc())
    if ARCHIVE is not None: (ARCHIVE / 'prepare_cell_error.log').write_text(traceback.format_exc())
    raise
finally:
    if PREPARE_ENV and ARCHIVE is not None and (WORKDIR / 'logs').is_dir():
        saved_logs = ARCHIVE / ('setup_logs_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
        shutil.copytree(WORKDIR / 'logs', saved_logs, dirs_exist_ok=False)
        print('准备日志已归档到：', saved_logs)


**当前主线：训练练习已收束，已有阶段不用重跑。** 下一步见[Corrector机制实践](../studies/policy_generalization/corrector_mechanism_practice.md)。下文保留历史排错说明。

2026-11-05数据修复：发布元信息的文件编号与实际分片不符；813/818/825的实际文件是213/214/215。新准备入口会补齐分片，并在CPU上核对episode、帧数、动作窗口和padding。保留本次失败输出；请运行更新后的准备单元格，再运行inspect，无需删除旧缓存。

2026-11-05修复：CLI读取base配置时显式使用预下载的固定SHA，避免离线模式默认查找main失败。已保留旧报错输出；重跑前请先执行更新后的准备单元格，把新SOURCE写入远程脚本，再运行inspect。无需清空缓存。

准备单元格保留了上次T4成功运行的输出；它不是当前连接状态。下次选择T4内核后先运行准备单元格，再执行下面的阶段入口。

- `STAGE="inspect"`：创建完整策略、加载真实batch，保存图片和张量摘要；不反向、不更新。
- 核对batch后改为`STAGE="practice"`：显式同步并调用你写的`train_one_batch`一次，保存到独立的`practice_step`目录。
- 核对单步结果后改为`STAGE="train"`：重新加载初始策略，官方训练5步，保存/归档checkpoint并做重载推理比较。

每次只运行当前阶段，不要连续“全部运行”。三种阶段不会接着上一次练习权重自动续训。已有目录保留，遇到失败先读日志。
首轮训练episode为813/818；当前5步验证训练流程，未安排仿真成功率评测。日志/运行清单和检查batch会在Drive已挂载时保存。
本地更新用户函数后，需要更新Notebook中的`PRACTICE_SOURCE`（可用export-notebook导出）；仅编辑本地.py不会自动改变远程文件。


In [ ]:
# 每次只执行一个阶段：inspect → practice → train。先检查输出，再改STAGE。
# 本单元格是运行与记录代码；你的核心训练函数在train_one_batch.py里。
import json
import hashlib

STAGE = "train"  # inspect：只看真实batch；practice：你的函数更新1次；train：官方训练5步
assert STAGE in {"inspect", "practice", "train"}
PYTHON = WORKDIR / "venv/bin/python"
assert PYTHON.is_file() and SCRIPT.is_file(), "先连接T4，并运行准备单元格"
assert SCRIPT.read_text() == SOURCE, "远程工作脚本不是当前版本，请重新运行准备单元格"
manifest = WORKDIR / "environment.json"
assert manifest.is_file(), "准备尚未完成，缺少environment.json"
if ARCHIVE is not None and str(ARCHIVE).startswith("/content/drive/"):
    assert os.path.ismount("/content/drive"), "Drive未挂载，先恢复准备步骤"

log_root = (ARCHIVE if ARCHIVE is not None else WORKDIR) / "stage_logs"
control_dir = log_root / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ") + "_" + STAGE + "_launch")
control_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(manifest, control_dir / "environment.json")
data_check = WORKDIR / "dataset_check.json"
if data_check.is_file(): shutil.copy2(data_check, control_dir / "dataset_check.json")
command = [str(PYTHON), str(SCRIPT), "train", "--workdir", str(WORKDIR),
           "--log-root", str(log_root)]
if ARCHIVE is not None:
    command += ["--archive-dir", str(ARCHIVE)]
if STAGE == "inspect":
    command += ["--inspect-only"]
    result_dir = WORKDIR / "batch_inspection"
elif STAGE == "practice":
    # STAGE显式选择practice时，同步本Notebook携带的最新版用户函数；旧远程文件先备份。
    if PRACTICE_FILE.exists() and PRACTICE_FILE.read_text() != PRACTICE_SOURCE:
        shutil.copy2(PRACTICE_FILE, control_dir / "previous_train_one_batch.py")
    PRACTICE_FILE.write_text(PRACTICE_SOURCE)
    print("使用你的函数：", PRACTICE_FILE, "SHA256:", hashlib.sha256(PRACTICE_FILE.read_bytes()).hexdigest()[:12])
    command += ["--practice-step-file", str(PRACTICE_FILE)]
    result_dir = (ARCHIVE if ARCHIVE is not None else WORKDIR) / "practice_step"
else:
    result_dir = WORKDIR / "train_run"
assert not result_dir.exists(), f"保留已有结果：{result_dir}；不要删除后盲目重跑，先核对该目录"

print("阶段：", STAGE, "；命令：", command)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
if returncode:
    raise RuntimeError(f"阶段{STAGE}失败，退出码{returncode}；完整日志：{log_root}")
if STAGE == "inspect":
    report = json.loads((result_dir / "batch_inspection.json").read_text())
    shutil.copytree(result_dir, control_dir / "batch_inspection")
    print(json.dumps(report["first_processed_batch"], ensure_ascii=False, indent=2))
else:
    report = json.loads((result_dir / "training_evidence.json").read_text())
    print(json.dumps(report["steps"], ensure_ascii=False, indent=2))
print("结果目录：", result_dir, "；本阶段日志：", log_root)
if ARCHIVE is None:
    print("未指定云盘归档；当前结果/日志只在本运行时，请另行保存。")


In [ ]:
import torch 
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

## 验证集预测导出：指标没写完，也能先保存GPU结果

当前会话先运行上面的准备单元格（`PREPARE_ENV=False`，只同步代码），**跳过STAGE训练单元格**，然后只运行下面这一格。新运行时先用True恢复环境。

加载已经保存的官方五步checkpoint，对未训练的episode 825的三个时刻，用相同输入和初始噪声比较LoRA禁用/启用的动作预测。首个时刻额外复算以检查重复性。不训练、不调用LIBERO、不需要`masked_action_mae`已实现。

每个样本写入Drive并更新进度；只有`evaluation_manifest.json`的`complete=true`才表示完整导出。随后可以释放GPU，在本地补写`studies/policy_generalization/evaluation_metrics.py`，运行`check-metric`和`score-eval`。下载时保留`paired_predictions.pt`与同目录的manifest；这一步没有机器人执行视频。


In [ ]:
# 只导出离线预测。已完成的inspect/practice/train不必重跑。
import json
import subprocess
from pathlib import Path

EVAL_RUN = "heldout_825_001"  # 已有目录不覆盖；再次实验请更换名字
assert SCRIPT.read_text() == SOURCE, "先执行更新后的代码同步单元格"
assert (WORKDIR / "smolvla_eval_tools.py").read_text() == EVAL_SOURCE, "评测工具尚未同步"
assert ARCHIVE is not None, "先挂载Drive，避免GPU结束后丢失预测"
python = WORKDIR / "venv/bin/python"
assert python.is_file() and (WORKDIR / "environment.json").is_file(), "请先准备环境"
result_dir = ARCHIVE / "evaluations" / EVAL_RUN
assert not result_dir.exists(), f"保留已有结果：{result_dir}；请更换EVAL_RUN"
command = [str(python), str(SCRIPT), "export-eval",
           "--workdir", str(WORKDIR), "--archive-dir", str(ARCHIVE),
           "--log-root", str(ARCHIVE / "stage_logs"), "--eval-run", EVAL_RUN]
# CLI自己将完整日志写入Drive，控制台同步显示真实错误。
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
if code:
    raise RuntimeError(f"导出失败，退出码{code}；请查看上方直接错误与{ARCHIVE / 'stage_logs'}，不要重新训练")
manifest = json.loads((result_dir / "evaluation_manifest.json").read_text())
assert manifest.get("complete") is True, "导出尚未完成，不能报告完整评测"
print("完整预测已保存：", result_dir)
print("验证样本：", manifest["samples"])
print("现在可以释放GPU，在本地CPU完成masked_action_mae和评分。")


## 保存checkpoint的LIBERO闭环执行与录像

**11月5日更新：本节保留作可选工程验证，不再作为Corrector的前置任务。** 当前训练练习已完成；下一步见[Corrector机制实践](../studies/policy_generalization/corrector_mechanism_practice.md)。

本次旧输出记录了安装后 `No module named libero` 的真实失败，保留供排查。更新后的准备单元格在已有环境中选择 `repair-sim`，注册源码路径并复查真实渲染；不重训、不重装。渲染检查录像不等于VLA任务执行录像。

下一次T4只运行：**最前面的环境准备/代码同步 → 本节的仿真准备 → 本节的评测 → 视频显示**。跳过旧inspect/practice/train/离线导出，它们已经完成。

仿真使用独立`sim-venv`，不修改已跑通的训练venv；加载原始LIBERO固定版本，不带view扰动后缀。准备阶段先做真实场景reset、两路图像/8维状态、MP4编码检查，不加载VLA。

成对评测加载Drive中的官方五步adapter与保存的processor，对同一alphabet soup任务、初始状态序列、seed，分别禁用/启用LoRA。默认各一回合，20Hz控制、20FPS录像，最多280步。数据10FPS与仿真控制频率的转换关系还未确认，因此这是初次执行协议检查，不是正式泛化评测。

成功/失败均录像，每回合立即保存到Drive；断连时保留部分结果。查看`eval_info.json`的`complete`与实际回合数，不能把缺失回合计作成功或失败。本地指标结果见`studies/policy_generalization/offline_eval_results_20261105.md`；核心解释见`libero_evaluation_practice.md`。


In [ ]:
# 先运行最前面的准备/同步单元格。这里只安装独立仿真依赖并做真实渲染检查。
import subprocess
assert SCRIPT.read_text() == SOURCE, "请先运行更新后的代码同步单元格"
assert (WORKDIR / "smolvla_libero_tools.py").read_text() == SIM_SOURCE, "仿真接入工具尚未同步"
assert ARCHIVE is not None, "先挂载Drive，保证完整日志持久化"
# 上次安装已完成但导入路径失败时，只修复路径并复查；不重复安装。
SIM_SETUP_MODE = "repair-sim" if (
    (WORKDIR / "sim-venv/bin/python").is_file()
    and (WORKDIR / "libero-original/libero/libero/__init__.py").is_file()
    and (WORKDIR / "libero-config/config.yaml").is_file()
) else "prepare-sim"
print("本次仿真准备模式：", SIM_SETUP_MODE)
command = [str(WORKDIR / "venv/bin/python"), str(SCRIPT), SIM_SETUP_MODE,
           "--workdir", str(WORKDIR), "--archive-dir", str(ARCHIVE),
           "--log-root", str(ARCHIVE / "simulation_setup_logs")]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
if code:
    raise RuntimeError(f"仿真准备失败，退出码{code}；请看上方直接错误，日志在{ARCHIVE / 'simulation_setup_logs'}")
print("仿真真实渲染/录像检查通过；还没有执行VLA任务评测。")
import json
from pathlib import Path
from IPython.display import Image as DisplayImage, display
sim_status = json.loads((WORKDIR / "simulation.json").read_text())
probe_reports = sorted(Path(sim_status["render_probe"]).glob("*/render_check.json"))
assert probe_reports, "缺少本次真实渲染检查报告"
print("本次渲染检查：", probe_reports[-1])
for name in ["observation_images_image.png", "observation_images_image2.png"]:
    display(DisplayImage(filename=str(probe_reports[-1].parent / name), width=256))
print("先核对相机朝向和场景外观；截图是策略输入空间的图像，不是VLA执行结果。")


In [ ]:
# 第一次只跑base/LoRA各1回合，不重新训练。
import json
import subprocess
RUN_NAME = "libero_checkpoint_smoke_001"  # 已有结果不覆盖
EPISODES = 1
CONTROL_FPS = 20  # 与录像FPS相同；不从训练数据10FPS直接推断
CONDITION = "both"  # both：成对；trained：只检查训练后策略的执行
assert SCRIPT.read_text() == SOURCE, "代码版本未同步"
assert ARCHIVE is not None and (WORKDIR / "simulation.json").is_file(), "先完成仿真准备"
SIM_RESULT_DIR = ARCHIVE / "libero_evaluations" / RUN_NAME
assert not SIM_RESULT_DIR.exists(), f"保留已有结果：{SIM_RESULT_DIR}，请更换RUN_NAME"
command = [str(WORKDIR / "sim-venv/bin/python"), str(SCRIPT), "eval-libero",
           "--workdir", str(WORKDIR), "--archive-dir", str(ARCHIVE),
           "--log-root", str(ARCHIVE / "simulation_eval_logs"), "--eval-run", RUN_NAME,
           "--episodes", str(EPISODES), "--sim-fps", str(CONTROL_FPS),
           "--condition", CONDITION, "--seed", "1000", "--max-steps", "280"]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, errors="replace") as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
if code:
    raise RuntimeError(f"评测失败，退出码{code}；保留{SIM_RESULT_DIR}中的已完成回合与完整日志，不需要重训")
result = json.loads((SIM_RESULT_DIR / "eval_info.json").read_text())
assert result["complete"] is True
print("实际完成回合数：", len(result["results"]))
print("成功率（0～1）：", result["success_rates"])
print("视频和逐回合记录：", SIM_RESULT_DIR)


In [ ]:
# 显示实际完成的视频；失败回合也保留。即使评测部分中断，也可手动指定SIM_RESULT_DIR查看已有回合。
from IPython.display import Video, display
import json
result = json.loads((SIM_RESULT_DIR / "eval_info.json").read_text())
print("完整评测：", result["complete"])
for row in result["results"]:
    print(row["condition"], "seed=", row["seed"], "success=", row["success"])
    for video in row["videos"]:
        print("控制/播放FPS：", result["control_fps"], video["playback_fps"])
        display(Video(video["path"], embed=True, width=512))
